# Building the data for the figures

This notebook makes every file in `data/` that the figure notebooks read, except `s4_figure_data.pkl`, which `s4_nll_build_data.ipynb` makes. It starts from the per-bird syllable tables, BioSound features, mutual information bootstraps, VAE results and wav recordings on the Gentner lab cluster, so it will not run anywhere else. It is here to show how each file was made; the saved output under each cell is from the run that made the files in `data/`.

**Run order:** top to bottom. Cells 1 to 4 and F1 are shared. Cell A1 (sound level) writes `syllable_level_data.pkl`, which Cells A3 to A5 and E2 read, so it runs first in its section.

| Section | Cells | Writes | Read by |
|---|---|---|---|
| Fig. 1B | F1, U1, U2 | `umap_examples_data.pkl`, `umap_example_spectrograms.pkl` | `umap_examples_figures` |
| Fig. 1 C-D, Fig. 2 B and D | X1 to X3 | `spectrogram_examples_data.pkl` | `spectrogram_examples_figures` |
| Fig. 2 A, C, G-J, Table 1, Supp. Fig. S2 | A1 to A6 | `syllable_level_data.pkl`, `syllable_features_data.pkl` | `syllable_features_figures` |
| Fig. 3 | T1 | `transition_structure_data.pkl` | `transition_structure_figures` |
| Fig. 4 C-G | SS1 | `song_structure_data.pkl` | `song_structure_figures` |
| Fig. 4 H-I, Supp. Fig. S5 | K1 to K5 | `kgram_data.pkl` | `kgram_figures` |
| Fig. 5 A-E, Supp. Fig. S6 | W1 to W4 | `within_song_data.pkl` | `within_song_figures` |
| Fig. 5 F-H | L1 | `vae_latent_data.pkl` | `vae_latent_figures` |
| Fig. 5 I-M, Supp. Fig. S7, Supp. Table S1 | E1 to E5 | `exposure_course_data.pkl` | `exposure_course_figures` |
| Supp. Fig. S1 | C1 to C3 | `clustering_validation_data.pkl` | `clustering_validation_figures` |
| Supp. Fig. S3 | M1 | `mutual_information_data.pkl` | `mutual_information_figures` |
| Supp. Fig. S4 | R1 | `repetition_data.pkl` | `repetition_figures` |

## Setup (Cells 1 to 4)

In [1]:
# Cell 1: paths
# Every input lives on the lab cluster; every output is written to DERIVED.
import os


def _path(env_name, default):
    # Each location can be overridden from the environment: export HELIOX_PROC_DIR=/somewhere/else
    return os.environ.get(env_name, default)


PROC_DIR = _path('HELIOX_PROC_DIR',
                 '/mnt/cube/lstanwic/umap_avgn_shared/lauren_umap/heliox_processing')
BASE_DIR = os.path.join(PROC_DIR, 'data')
DERIVED = os.path.join(BASE_DIR, 'derived')      # every *_data.pkl the figure notebooks read is saved here
STARLING_SYL_DIR = os.path.join(PROC_DIR, 'figures', 'starling_runs')
ZF_SYL_PATH = os.path.join(BASE_DIR, 'zf_combined', 'syldf_fullycombined_20250807.pickle')
MOTIF_PKL = os.path.join(DERIVED, 'starling_final_motif_sets.pkl')
os.makedirs(DERIVED, exist_ok=True)
print('derived data ->', DERIVED)

derived data -> /mnt/cube/lstanwic/umap_avgn_shared/lauren_umap/heliox_processing/data/derived


In [2]:
# Cell 2: analysis parameters
# Rationale: the settings shared by several sections live here, so that every section uses the same ones.
LABCOL = 'hdbscan_labels_new_0_005'   # 0.5% min_cluster_size, the clustering in the paper
BOUTCOL = 'bout_index'
POSCOL = 'syllables_sequence_pos'

# starling names as they appear in the syllable tables (and in the k-gram sets)
STARLING_BIRDS = ['s_b1556h_24', 's_b1484h_24', 's_b1364_25',
                  's_b1354_25', 's_b1481h_24', 's_b1349h_23']

# Every table must hold this many birds in BOTH conditions. A bird whose model file or wav
# folder fails to resolve would otherwise drop out silently and the models would still fit.
EXPECTED_BIRDS = {'starling': 6, 'zebra_finch': 6}
# False: stop when a table is short of birds. True: say so loudly, record it in the
# pickle, and carry on. Leave it False unless a shortfall is understood and intended.
ALLOW_INCOMPLETE = False

COND_ORDER = ['regair', 'heliox']

In [3]:
# Cell 3: imports and shared helpers
import re, pickle, datetime
import numpy as np
import pandas as pd
from scipy.io.wavfile import read as wavread


class _NumpyCompatUnpickler(pickle.Unpickler):
    """Redirects numpy._core.* <-> numpy.core.* during unpickling, so results saved under
    one numpy major version load under the other."""
    def find_class(self, module, name):
        if module.startswith('numpy._core'):
            module = module.replace('numpy._core', 'numpy.core', 1)
        elif module.startswith('numpy.core'):
            alt = module.replace('numpy.core', 'numpy._core', 1)
            try:
                return super().find_class(alt, name)
            except ModuleNotFoundError:
                pass
        return super().find_class(module, name)


def load_pickle(path):
    with open(path, 'rb') as f:
        try:
            return pickle.load(f)
        except ModuleNotFoundError as e:
            if '_core' not in str(e) and 'numpy' not in str(e):
                raise
            f.seek(0)
            return _NumpyCompatUnpickler(f).load()


def bird_num(s):
    """'s_b1556h_24' -> '1556'; also works on S4 folder names."""
    m = re.search(r'b(\d+)', str(s))
    return m.group(1) if m else None




INCOMPLETE = []     # filled only when ALLOW_INCOMPLETE lets a short table through


def require_complete(df, name, bird_col, cond_col, species=None, species_col=None):
    """Stop if any species x condition cell is short of birds. `species` names the species
    of a single-species table; `species_col` is the column in a two-species one."""
    problems = []
    for sp, n_expected in EXPECTED_BIRDS.items():
        if species is not None and sp != species:
            continue
        sub = df if species_col is None else df[df[species_col] == sp]
        for cond in COND_ORDER:
            n = sub.loc[sub[cond_col] == cond, bird_col].nunique()
            if n != n_expected:
                problems.append(f'{sp} {cond}: {n} birds, expected {n_expected}')
    if not problems:
        print(f'{name}: complete ({len(df)} rows)')
        return
    message = f'{name} is missing birds -> ' + '; '.join(problems)
    if not ALLOW_INCOMPLETE:
        raise RuntimeError(message + '. Set ALLOW_INCOMPLETE = True in Cell 2 to continue anyway.')
    INCOMPLETE.append(message)
    print('[INCOMPLETE]', message)


print('helpers ready')

helpers ready


In [4]:
# Cell 4: shared check before saving
def assert_portable(name, df):
    """No path objects and no absolute paths: either would tie the pickle to this cluster."""
    for c in df.columns:
        if not (pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c])):
            continue
        vals = pd.Series(df[c].dropna().unique())
        if vals.map(lambda v: isinstance(v, os.PathLike)).any():
            raise TypeError(f'{name}.{c} holds path objects; store the file name as text')
        if vals.map(lambda v: isinstance(v, str) and os.sep in v).any():
            raise ValueError(f'{name}.{c} holds file paths; store the file name only')




## Fig. 1 and the example panels of Fig. 2

Cell F1 holds the tables and helpers shared by this section and by Supp. Fig. S1. U2 is only needed for the "run a new UMAP" example at the end of `umap_examples_figures`.

In [5]:
# Cell F1: tables and helpers shared by the Fig. 1, Fig. 2 B/D and Supp. Fig. S1 cells (U1, C1-C3, X1-X3)
import gc

# One syllable table per bird. These hold the UMAP embedding, the wav path and the onset and
# offset of every syllable, and are the tables the example and clustering notebooks read.
STARLING_FEATURE_FILES = {
    's_b1349h_23': 's_b1349h_23/syllable_df_labels_mindistpoint2_featureanalysismod_20250612.pickle',
    's_b1354_25':  's_b1354_25/syllable_df_labels_mindistpoint2_featureanalysismod_20250611.pickle',
    's_b1364h_25': 's_b1364h_25/syllable_df_labels_mindistpoint2_featureanalysismod_20250612.pickle',
    's_b1481h_24': 's_b1481h_24/syllable_df_labels_mindistpoint2_featureanalysismod_20250612.pickle',
    's_b1484h_24': 's_b1484h_24/syllable_df_labels_mindistpoint2_featureanalysismod_20250612.pickle',
    's_b1556h_24': 's_b1556h_24/syllable_df_labels_mindistpoint2_featureanalysismod_20250612.pickle',
}
# Same dict as Cell E2 and same folder as Cell W1, repeated so these cells run without them.
ZF_FEATURE_FILES = {
    'z_r11g11_24': 'z_r11g11_24/syllable_df_labels_featureanalysismod_20250807.pickle',
    'z_p7y2_23':   'z_p7y2_23/syllable_df_labels_featureanalysismod_20250611.pickle',
    'z_y11c13_24': 'z_y11c13_24/syllable_df_labels_featureanalysismod_20250611.pickle',
    'z_c8c19_24':  'z_c8c19_24/syllable_df_labels_featureanalysismod_20250807.pickle',
    'z_p4g13_23':  'z_p4g13_23/syllable_df_labels_featureanalysismod_20250807.pickle',
    'z_g14g16_24': 'z_g14g16_24/syllable_df_labels_featureanalysismod_20250804.pickle',
}
AUGMENTED_DIR = _path('HELIOX_STARLING_SYL_DIR', os.path.join(PROC_DIR, 'figures'))


def read_columns(path, wanted):
    """One syllable table cut down to the `wanted` columns, rows in their original order.
    The tables carry the audio and a spectrogram of every syllable, so the full table is let
    go straight away. Columns are taken by position, first occurrence of each name, since
    one table carries a column name twice."""
    raw = pd.read_pickle(path)
    cols = list(raw.columns)
    keep = [i for i, c in enumerate(cols) if c in wanted and c not in cols[:i]]
    out = raw.iloc[:, keep].copy()
    del raw
    gc.collect()
    return out


def umap_xy(df):
    """UMAP coordinates as two arrays. Most tables store umap_x and umap_y; the balanced
    table of bird 1364 stores one 'umap' column of [x, y] pairs."""
    if {'umap_x', 'umap_y'}.issubset(df.columns) and df[['umap_x', 'umap_y']].notna().all().all():
        return df['umap_x'].to_numpy(float), df['umap_y'].to_numpy(float)
    if 'umap' not in df.columns:
        raise KeyError(f'no UMAP coordinates (umap_x and umap_y, or umap) among the columns {list(df.columns)}')
    xy = np.vstack([np.asarray(v, dtype=float).ravel()[:2] for v in df['umap']])
    return xy[:, 0], xy[:, 1]


def wav_name(p):
    """File name of a wav path stored as text, as a Path, or as the text "PosixPath('...')"."""
    return os.path.basename(str(p).rstrip("')\""))


def wav_folder(p):
    """Name of the folder a wav sits in (the bird's folder under data/)."""
    return os.path.basename(os.path.dirname(str(p).rstrip("')\"")))


def py(v):
    """A numpy scalar as the plain Python value, so that printed settings can be pasted back."""
    return v.item() if hasattr(v, 'item') else v


print('Fig. 1 / Supp. S1 helpers ready')

Fig. 1 / Supp. S1 helpers ready


In [6]:
# Cell U1: UMAP examples (Fig. 1B) for umap_examples_figures.ipynb
# One representative bird per species: each syllable's UMAP position, syllable group and condition.
UMAP_EXAMPLES_DATA_PATH = _path('HELIOX_UMAP_EXAMPLES_DATA', os.path.join(DERIVED, 'umap_examples_data.pkl'))
UMAP_EXAMPLE_BIRDS = {'starling': 's_b1349h_23', 'zebra_finch': 'z_r11g11_24'}

frames = []
for species, bird in UMAP_EXAMPLE_BIRDS.items():
    if species == 'starling':
        # the augmented table carries the clustering used throughout the paper (LABCOL, -1 = unclustered)
        path, labcol = os.path.join(AUGMENTED_DIR, bird, f'{bird}_syllable_df_augmented.pkl'), LABCOL
    else:
        # zebra finch syllable types: k-means clusters in the UMAP embedding, labelled by hand
        path, labcol = os.path.join(BASE_DIR, ZF_FEATURE_FILES[bird]), 'label'
    df = read_columns(path, ['condition', 'umap', 'umap_x', 'umap_y', labcol])
    if labcol not in df.columns:
        raise KeyError(f'{bird}: no {labcol!r} column in {os.path.basename(path)}')
    x, y = umap_xy(df)
    number = pd.to_numeric(df[labcol], errors='coerce')
    one = pd.DataFrame({
        'species': species, 'bird': bird,
        'condition': df['condition'].astype(str).str.lower().to_numpy(),
        'label': (number.astype(int) if number.notna().all() else df[labcol].astype(str)).to_numpy(),
        'clustered': (number >= 0).to_numpy() if species == 'starling' else df[labcol].notna().to_numpy(),
        'umap_x': x, 'umap_y': y})
    # regular air and heliox only: transition bouts and regular air recorded after heliox are not shown
    other = one.loc[~one['condition'].isin(COND_ORDER), 'condition'].value_counts().to_dict()
    one = one[one['condition'].isin(COND_ORDER)].reset_index(drop=True)
    missing = [c for c in COND_ORDER if not (one['condition'] == c).any()]
    if missing or not np.isfinite(one[['umap_x', 'umap_y']].to_numpy()).all():
        raise ValueError(f'{bird}: no syllables in {missing}, or UMAP coordinates that are not numbers')
    n_groups = one.loc[one['clustered'], 'label'].nunique()
    print(f'{species:12s} {bird}: {len(one):>6d} syllables '
          f'({(one["condition"] == "regair").sum()} regair, {(one["condition"] == "heliox").sum()} heliox), '
          f'{n_groups} syllable groups, {int((~one["clustered"]).sum())} unclustered'
          + (f'   [left out: {other}]' if other else ''))
    frames.append(one)

umap_examples = {
    'syllables': pd.concat(frames, ignore_index=True),
    'meta': {'created': datetime.datetime.now().isoformat(timespec='seconds'),
             'birds': UMAP_EXAMPLE_BIRDS,
             'label': {'starling': f'{LABCOL}: HDBSCAN on the UMAP embedding, min_cluster_size 0.5% of the '
                                   "bird's syllables; clustered is False for unclustered syllables",
                       'zebra_finch': 'label: k-means cluster in the UMAP embedding, labelled by hand'},
             'versions': {'numpy': np.__version__, 'pandas': pd.__version__}},
}
assert_portable('syllables', umap_examples['syllables'])
with open(UMAP_EXAMPLES_DATA_PATH, 'wb') as f:
    pickle.dump(umap_examples, f, protocol=4)
print(f'saved {UMAP_EXAMPLES_DATA_PATH}  ({os.path.getsize(UMAP_EXAMPLES_DATA_PATH) / 1e6:.1f} MB)')

starling     s_b1349h_23:  12683 syllables (4451 regair, 8232 heliox), 47 syllable groups, 3389 unclustered
zebra_finch  z_r11g11_24:   6273 syllables (4041 regair, 2232 heliox), 7 syllable groups, 0 unclustered
saved /mnt/cube/lstanwic/umap_avgn_shared/lauren_umap/heliox_processing/data/derived/umap_examples_data.pkl  (0.7 MB)


In [7]:
# Cell U2 (optional): syllable spectrograms of the two example birds, for the "run a new UMAP" example
# at the end of umap_examples_figures.ipynb. Fig. 1B itself does not need this file.
# Run after F1 and U1 (same birds, same tables, same rows).
UMAP_SPECTROGRAMS_PATH = _path('HELIOX_UMAP_SPECTROGRAMS', os.path.join(DERIVED, 'umap_example_spectrograms.pkl'))
# the padded spectrogram of each syllable, which is what UMAP is run on
SPEC_COLUMN = {'starling': 'spectrogram', 'zebra_finch': 'processed_spec'}

umap_inputs = {}
for species, bird in UMAP_EXAMPLE_BIRDS.items():
    if species == 'starling':
        path, labcol = os.path.join(AUGMENTED_DIR, bird, f'{bird}_syllable_df_augmented.pkl'), LABCOL
    else:
        path, labcol = os.path.join(BASE_DIR, ZF_FEATURE_FILES[bird]), 'label'
    df = read_columns(path, ['condition', SPEC_COLUMN[species], labcol])
    if SPEC_COLUMN[species] not in df.columns:
        raise KeyError(f'{bird}: no {SPEC_COLUMN[species]!r} column in {os.path.basename(path)}')
    df['condition'] = df['condition'].astype(str).str.lower()
    df = df[df['condition'].isin(COND_ORDER)]
    shapes = df[SPEC_COLUMN[species]].map(np.shape).value_counts()
    if len(shapes) != 1:
        # UMAP needs one row of equal length per syllable, so the spectrograms must be padded to one shape
        raise ValueError(f'{bird}: {SPEC_COLUMN[species]} holds spectrograms of {len(shapes)} different shapes: '
                         f'{dict(shapes.head(3))}')
    spectrograms = np.stack(df[SPEC_COLUMN[species]].to_list())
    if spectrograms.dtype == np.float64:
        spectrograms = spectrograms.astype(np.float32)        # half the file, nothing lost for UMAP
    number = pd.to_numeric(df[labcol], errors='coerce')
    umap_inputs[species] = {'bird': bird, 'spectrograms': spectrograms, 'condition': df['condition'].to_numpy(),
                            'label': (number.astype(int) if number.notna().all() else df[labcol].astype(str)).to_numpy()}
    print(f'{species:12s} {bird}: {len(spectrograms)} syllables, each {spectrograms.shape[1]} x {spectrograms.shape[2]} '
          f'({spectrograms.dtype})')

umap_inputs['meta'] = {'created': datetime.datetime.now().isoformat(timespec='seconds'),
                       'spectrogram_column': SPEC_COLUMN,
                       'versions': {'numpy': np.__version__, 'pandas': pd.__version__}}
with open(UMAP_SPECTROGRAMS_PATH, 'wb') as f:
    pickle.dump(umap_inputs, f, protocol=4)
print(f'saved {UMAP_SPECTROGRAMS_PATH}  ({os.path.getsize(UMAP_SPECTROGRAMS_PATH) / 1e6:.0f} MB)')

starling     s_b1349h_23: 12683 syllables, each 32 x 27 (uint8)
zebra_finch  z_r11g11_24: 6273 syllables, each 64 x 22 (uint8)
saved /mnt/cube/lstanwic/umap_avgn_shared/lauren_umap/heliox_processing/data/derived/umap_example_spectrograms.pkl  (20 MB)


In [39]:
# Cell X1: spectrogram examples (Fig. 1 C-D, Fig. 2 B and D), which songs and syllables to draw
# Feeds spectrogram_examples_figures.ipynb.
SPEC_EXAMPLES_DATA_PATH = _path('HELIOX_SPEC_EXAMPLES_DATA', os.path.join(DERIVED, 'spectrogram_examples_data.pkl'))

# ---- Fig. 1C: one starling song per condition -------------------------------------------------
# song: the wav file name. An integer is the bout number the original example notebook used
# (order of first appearance in the bird's syllable table).
STARLING_SONGS = {'regair': dict(bird='s_b1349h_23', song='s_b1349h_23_2024-01-19_10-00-01-01_1729755_1750905_song_regair.wav'),
                  'heliox': dict(bird='s_b1349h_23', song='s_b1349h_23_2024-01-19_12-00-01-01_552590_576145_song_heliox.wav')}
STARLING_SONG_VIEW = dict(db_range=60, freq_khz=(0, 24))

# ---- Fig. 1D: one zebra finch song per condition, named by recording day and bout number ------
ZF_SONGS = {'regair': dict(bird='z_r11g11_24', sess='2025-02-17', bout_index=183),
            'heliox': dict(bird='z_r11g11_24', sess='2025-02-17', bout_index=237)}
ZF_SONG_VIEW = dict(db=(-55, 0), freq_khz=(0, 9), hop_ms=1, win_ms=5, n_fft=1024)
ZF_LETTERS = {0: 'I', 1: 'A', 2: 'B', 3: 'C', 4: 'D', 5: 'E', 6: 'F'}

# ---- Fig. 2 B and D: one entry per column, left to right --------------------------------------
# bird, label, offset, n, seed select the same candidate syllables as the original example cell.
# pick says which candidate is drawn in each condition (0 = the first candidate).
ZF_SYLLABLES = [
    dict(bird='z_p4g13_23', label=3, offset=10, n=10, seed=None, pad_ms=10, hop_ms=0.1, win_ms=5, n_fft=1024,
         db=(-40, -5), freq_khz=(0, 9), pick=dict(regair=2, heliox=0)),
    dict(bird=None, label=None, offset=10, n=10, seed=None, pad_ms=10, hop_ms=1, win_ms=5, n_fft=1024,
         db=(-40, 0), freq_khz=(0, 9), pick=dict(regair=0, heliox=0)),      # middle column: not identified yet
    dict(bird='z_r11g11_24', label=5, offset=10, n=10, seed=None, pad_ms=10, hop_ms=1, win_ms=5, n_fft=1024,
         db=(-40, 0), freq_khz=(0, 8.5), pick=dict(regair=8, heliox=5)),
]
STARLING_SYLLABLES = [
    dict(bird='s_b1556h_24', label=22, offset=3, n=10, seed=None, pad_ms=20, db_range=60, freq_khz=(0, 21),
         pick=dict(regair=2, heliox=1)),
    dict(bird='s_b1481h_24', label=0, offset=5, n=10, seed=None, pad_ms=20, db_range=65, freq_khz=(0, 20.001),
         pick=dict(regair=2, heliox=1)),
    dict(bird='s_b1349h_23', label=1, offset=5, n=10, seed=23, pad_ms=50, db_range=15, freq_khz=(0.05, 22),
         pick=dict(regair=6, heliox=1)),
]
# True once the picks above have been checked against the syllables in the paper.
PICKS_CONFIRMED = True

print('spectrogram examples ->', SPEC_EXAMPLES_DATA_PATH)

spectrogram examples -> /mnt/cube/lstanwic/umap_avgn_shared/lauren_umap/heliox_processing/data/derived/spectrogram_examples_data.pkl


In [40]:
# Cell X2: syllable tables, audio and the two spectrogram pipelines behind the example panels
import sys
import librosa
sys.path.append(PROC_DIR)                                  # where the lab copy of starling_rhythm lives
from vocalseg.dynamic_thresholding import dynamic_threshold_segmentation
from avgn.signalprocessing.filtering import butter_bandpass_filter
try:
    from starling_rhythm.utils.audio import int16_to_float32
except ImportError:                                        # AVGN ships the same function
    from avgn.utils.audio import int16_to_float32

# Display settings of the starling example spectrograms. These are not the settings the songs
# were segmented with; they only decide how the example panels look.
STARLING_SPEC = dict(min_level_db=-80, min_level_db_floor=-40, db_delta=5, n_fft=1024, hop_length_ms=1,
                     win_length_ms=5, ref_level_db=20, pre=0.97, silence_threshold=0.0,
                     max_vocal_for_spec=2.0, min_syllable_length_s=0.075)
STARLING_BANDPASS_HZ = (500, 15000)

_starling_tables, _zf_tables, _zf_audio, _last_song = {}, {}, {}, {}


def starling_table(bird):
    """A starling's syllables in the row order of its table (the order the examples were picked in)."""
    if bird not in _starling_tables:
        df = read_columns(os.path.join(BASE_DIR, STARLING_FEATURE_FILES[bird]),
                          ['file', 'bout_index', 'condition', 'start_time', 'end_time', 'hdbscan_labels'])
        t = pd.DataFrame({'bird': bird, 'condition': df['condition'].astype(str).str.lower().to_numpy(),
                          'song_file': df['file'].map(wav_name).to_numpy(),
                          'folder': df['file'].map(wav_folder).to_numpy(),
                          'bout': df['bout_index' if 'bout_index' in df.columns else 'file'].map(str).to_numpy(),
                          'onset_s': df['start_time'].to_numpy(float), 'offset_s': df['end_time'].to_numpy(float),
                          'label': df['hdbscan_labels'].to_numpy()})
        t = t[t['condition'] != 'regairb'].reset_index(drop=True)
        t['song_number'] = pd.factorize(t['bout'])[0]       # the bout numbers of the original example notebook
        _starling_tables[bird] = t.drop(columns='bout')
    return _starling_tables[bird]


def zf_table(bird):
    """A zebra finch's syllables in the row order of its table. A bout is a recording day (sess) plus a
    bout number, since bout numbers start again each day."""
    if bird not in _zf_tables:
        df = read_columns(os.path.join(BASE_DIR, ZF_FEATURE_FILES[bird]),
                          ['sess', 'bout_index', 'condition', 'on_ms', 'off_ms', 'label'])
        df['condition'] = df['condition'].astype(str).str.lower()
        df['bird'] = bird
        _zf_tables[bird] = df[df['condition'] != 'transition'].reset_index(drop=True)
    return _zf_tables[bird]


def zf_bout_audio(bird, bouts):
    """{(sess, bout_index): (filtered waveform, sample rate)} for the bouts asked for. Reading the full
    table is slow (it carries every bout's audio), so each bout is kept once it has been read."""
    bouts = [tuple(b) for b in bouts]
    need = sorted(set(b for b in bouts if (bird,) + b not in _zf_audio))
    if need:
        raw = pd.read_pickle(os.path.join(BASE_DIR, ZF_FEATURE_FILES[bird]))
        first = raw.drop_duplicates(['sess', 'bout_index']).set_index(['sess', 'bout_index'])
        for b in need:
            row = first.loc[b]
            _zf_audio[(bird,) + b] = (np.asarray(row['bout_waveform_filt']).flatten().astype(np.float32),
                                      int(row['sample_rate']))
        del raw, first
        gc.collect()
    return {b: _zf_audio[(bird,) + b] for b in bouts}


def starling_bout_db(folder, song_file):
    """One starling song as (dB spectrogram, duration in s, top frequency in Hz, white point in dB).
    Band-pass, then the vocalseg spectrogram (normalised, median removed per frequency), then dB."""
    # only the latest song is kept: one spectrogram is ~100 MB, and neighbouring candidates share a song
    if _last_song.get('key') != (folder, song_file):
        rate, data = wavread(os.path.join(BASE_DIR, folder, song_file))
        data = butter_bandpass_filter(int16_to_float32(data), STARLING_BANDPASS_HZ[0], STARLING_BANDPASS_HZ[1], rate)
        seg = dynamic_threshold_segmentation(data, rate, verbose=False, **STARLING_SPEC)
        if seg is None:
            raise RuntimeError(f'{song_file}: no spectrogram threshold gave a usable envelope')
        db = librosa.power_to_db(seg['spec'], ref=np.max)
        _last_song.update(key=(folder, song_file),
                          value=(db, len(data) / rate, rate / 2.0, float(np.percentile(db, 99.8))))
    return _last_song['value']


def zf_bout_db(y, sr, hop_ms, win_ms, n_fft, pre=0.97):
    """One zebra finch bout as (dB spectrogram relative to its peak, duration in s, top frequency in Hz)."""
    y = np.append(y[0], y[1:] - pre * y[:-1])               # pre-emphasis: lifts the high frequencies
    hop = max(1, int(hop_ms * sr / 1000))
    S = librosa.stft(y, n_fft=n_fft, hop_length=hop, win_length=max(1, int(win_ms * sr / 1000)))
    db = librosa.amplitude_to_db(np.abs(S), ref=np.max)
    return db, db.shape[1] * hop / sr, sr / 2.0


def cut(db, dur_s, top_hz, t0, t1, freq_khz, db_floor, db_ceiling):
    """The [t0, t1] s by freq_khz window of a spectrogram as a panel record. dB between floor and
    ceiling are stored as 0-255: a grey-scale panel shows no more than that, and the file stays small."""
    n_f, n_t = db.shape
    sec = dur_s / n_t
    c0 = max(0, int(t0 / sec))
    c1 = max(c0 + 1, min(n_t, int(t1 / sec) + 1))
    f = np.linspace(0, top_hz, n_f)
    rows = np.where((f >= freq_khz[0] * 1000) & (f <= freq_khz[1] * 1000))[0]
    window = np.clip(db[rows[0]:rows[-1] + 1, c0:c1], db_floor, db_ceiling)
    return dict(image=np.round((window - db_floor) / (db_ceiling - db_floor) * 255).astype(np.uint8),
                t0=c0 * sec, t1=c1 * sec, f0_khz=f[rows[0]] / 1000, f1_khz=f[rows[-1]] / 1000,
                db_floor=float(db_floor), db_ceiling=float(db_ceiling))


def candidates(table, ex, cond, rng):
    """The syllables the original example cell showed for one syllable type and condition, in its
    order: the first n after `offset`, or the first n of a seeded shuffle."""
    rows = table[(table['label'] == ex['label']) & table['condition'].str.contains(cond)].reset_index(drop=True)
    if rng is not None and len(rows):
        return rows.iloc[rng.permutation(len(rows))].reset_index(drop=True).iloc[:ex['n']]
    start = min(ex['offset'], max(0, len(rows) - ex['n']))
    return rows.iloc[start:start + ex['n']].reset_index(drop=True)


def syllable_panels(species, ex, picked_only=False):
    """{condition: list of panel records} for the candidate syllables of one Fig. 2 B/D column,
    or for the picked one alone."""
    rng = None if ex['seed'] is None else np.random.default_rng(ex['seed'])   # one generator, regair first
    table = starling_table(ex['bird']) if species == 'starling' else zf_table(ex['bird'])
    pad = ex['pad_ms'] / 1000.0
    out = {}
    for cond in COND_ORDER:
        rows = candidates(table, ex, cond, rng)
        if picked_only:
            if ex['pick'][cond] >= len(rows):
                raise IndexError(f'{ex["bird"]} {cond}: pick {ex["pick"][cond]}, but there are {len(rows)} candidates')
            rows = rows.iloc[[ex['pick'][cond]]]
        panels = []
        if species == 'starling':
            for _, r in rows.iterrows():
                db, dur, top, white = starling_bout_db(r['folder'], r['song_file'])
                p = cut(db, dur, top, r['onset_s'] - pad, r['offset_s'] + pad, ex['freq_khz'],
                        white - ex['db_range'], white)
                p.update(source=r['song_file'], onset_s=float(r['onset_s']), offset_s=float(r['offset_s']))
                panels.append(p)
        else:
            audio = zf_bout_audio(ex['bird'], zip(rows['sess'], rows['bout_index']))
            other_day = 0
            for _, r in rows.iterrows():
                y, sr = audio[(r['sess'], r['bout_index'])]
                db, dur, top = zf_bout_db(y, sr, ex['hop_ms'], ex['win_ms'], ex['n_fft'])
                on, off = r['on_ms'] / 1000.0, r['off_ms'] / 1000.0
                p = cut(db, dur, top, on - pad, off + pad, ex['freq_khz'], *ex['db'])
                p.update(source=f'bout {py(r["bout_index"])} of {r["sess"]}', onset_s=on, offset_s=off)
                # The original example cell looked a bout up by its number alone. Count the syllables for
                # which that finds an earlier day's bout: their published picture would be from that bout.
                other_day += table.loc[table['bout_index'] == r['bout_index'], 'sess'].iloc[0] != r['sess']
                panels.append(p)
            if other_day:
                print(f'  [CHECK] {ex["bird"]} type {ex["label"]} {cond}: {other_day} of these {len(rows)} '
                      'syllables sit in a bout whose number was also used on an earlier day. '
                      'The original cell drew those from the earlier bout.')
        out[cond] = panels
    return out


print('spectrogram pipelines ready')

spectrogram pipelines ready


In [41]:
# Cell X3: build the panels, check, save
x_images, x_rows, x_song_syllables = {}, [], []


def add_panel(figure, column, cond, species, bird, label, p):
    panel_id = f'{figure}_{column}_{cond}'
    x_images[panel_id] = p['image']
    x_rows.append(dict(panel_id=panel_id, figure=figure, column=column, condition=cond, species=species,
                       bird=bird, source=p['source'], label='' if label is None else str(label),
                       onset_s=p.get('onset_s', np.nan), offset_s=p.get('offset_s', np.nan),
                       t0=p['t0'], t1=p['t1'], f0_khz=p['f0_khz'], f1_khz=p['f1_khz'],
                       db_floor=p['db_floor'], db_ceiling=p['db_ceiling']))
    return panel_id


# ---- Fig. 1C: starling songs -------------------------------------------------------------------
for cond, choice in STARLING_SONGS.items():
    if choice['song'] is None:
        print(f'[TO DO] Fig. 1C {cond}: no song chosen in Cell X1.')
        continue
    t = starling_table(choice['bird'])
    by_number = isinstance(choice['song'], (int, np.integer))        # a bout number of the original notebook
    syl = t[t['song_number'] == choice['song']] if by_number else t[t['song_file'] == os.path.basename(choice['song'])]
    if syl.empty or set(syl['condition']) != {cond}:
        raise KeyError(f'Fig. 1C {cond}: {choice["song"]!r} is not a {cond} song of {choice["bird"]}')
    db, dur, top, white = starling_bout_db(syl['folder'].iloc[0], syl['song_file'].iloc[0])
    start = max(0.0, choice.get('start_s', 0.0))                     # a panel may begin partway into the wav
    p = cut(db, dur, top, start, dur, STARLING_SONG_VIEW['freq_khz'], white - STARLING_SONG_VIEW['db_range'], white)
    start = p['t0']
    p.update(source=syl['song_file'].iloc[0], t0=0.0, t1=p['t1'] - start)     # time runs from 0, as in the paper
    panel_id = add_panel('1C', 1, cond, 'starling', choice['bird'], None, p)
    syl = syl[syl['offset_s'] > start]
    x_song_syllables.append(pd.DataFrame({'panel_id': panel_id, 'onset_s': syl['onset_s'].to_numpy() - start,
                                          'offset_s': syl['offset_s'].to_numpy() - start,
                                          'label': syl['label'].to_numpy(), 'letter': ''}))

# ---- Fig. 1D: zebra finch songs ----------------------------------------------------------------
for cond, choice in ZF_SONGS.items():
    if choice['bird'] is None:
        print(f'[TO DO] Fig. 1D {cond}: ZF_SONGS is not filled in (Cell X1).')
        continue
    bout = (choice['sess'], choice['bout_index'])
    t = zf_table(choice['bird'])
    syl = t[(t['sess'] == bout[0]) & (t['bout_index'] == bout[1])].sort_values('on_ms')
    if syl.empty or set(syl['condition']) != {cond}:
        raise KeyError(f'Fig. 1D {cond}: bout {bout[1]!r} of {bout[0]!r} is not a {cond} bout of {choice["bird"]}')
    y, sr = zf_bout_audio(choice['bird'], [bout])[bout]
    db, dur, top = zf_bout_db(y, sr, ZF_SONG_VIEW['hop_ms'], ZF_SONG_VIEW['win_ms'], ZF_SONG_VIEW['n_fft'])
    p = cut(db, dur, top, 0, dur, ZF_SONG_VIEW['freq_khz'], *ZF_SONG_VIEW['db'])
    p['source'] = f'bout {bout[1]} of {bout[0]}'
    panel_id = add_panel('1D', 1, cond, 'zebra_finch', choice['bird'], None, p)
    x_song_syllables.append(pd.DataFrame({'panel_id': panel_id, 'onset_s': syl['on_ms'].to_numpy() / 1000.0,
                                          'offset_s': syl['off_ms'].to_numpy() / 1000.0,
                                          'label': syl['label'].to_numpy(),
                                          'letter': [ZF_LETTERS.get(py(v), '') for v in syl['label']]}))

# ---- Fig. 2 B and D: one syllable per condition and column ---------------------------------------
for figure, species, examples in [('2B', 'zebra_finch', ZF_SYLLABLES), ('2D', 'starling', STARLING_SYLLABLES)]:
    for column, ex in enumerate(examples, start=1):
        if ex['bird'] is None:
            print(f'[TO DO] Fig. {figure} column {column}: no bird and label in Cell X1 yet')
            continue
        panels = syllable_panels(species, ex, picked_only=True)
        for cond in COND_ORDER:
            add_panel(figure, column, cond, species, ex['bird'], ex['label'], panels[cond][0])
if not PICKS_CONFIRMED:
    print('[TO DO] the Fig. 2 B/D picks are best guesses from the saved notebook output. Check them '
          'against the paper, then set PICKS_CONFIRMED = True in Cell X1.')

x_panels = pd.DataFrame(x_rows)
song_syllables = (pd.concat(x_song_syllables, ignore_index=True) if x_song_syllables else
                  pd.DataFrame(columns=['panel_id', 'onset_s', 'offset_s', 'label', 'letter']))
assert_portable('panels', x_panels)
assert_portable('song_syllables', song_syllables)
print(x_panels[['panel_id', 'bird', 'label', 'source']].to_string(index=False))

spec_examples = {
    'panels': x_panels, 'images': x_images, 'song_syllables': song_syllables,
    'meta': {
        'created': datetime.datetime.now().isoformat(timespec='seconds'),
        'picks_confirmed': bool(PICKS_CONFIRMED),
        'settings': dict(STARLING_SPEC=STARLING_SPEC, STARLING_BANDPASS_HZ=STARLING_BANDPASS_HZ,
                         STARLING_SONG_VIEW=STARLING_SONG_VIEW, ZF_SONG_VIEW=ZF_SONG_VIEW,
                         STARLING_SYLLABLES=STARLING_SYLLABLES, ZF_SYLLABLES=ZF_SYLLABLES),
        'notes': {'images': 'grey levels 0-255 spanning db_floor to db_ceiling; row 0 is the lowest frequency (f0_khz)',
                  'starling': 'band-pass 0.5-15 kHz, vocalseg spectrogram (1 ms hop, 5 ms window, median removed '
                              'per frequency), dB; white point = 99.8th percentile of the song',
                  'zebra_finch': 'pre-emphasis 0.97, STFT, dB relative to the bout peak',
                  'song_syllables': 'syllable onsets and offsets (s) of the Fig. 1 C-D songs, for the bars under them',
                  'source': 'wav file name (starling) or bout number and recording day (zebra finch)'},
        'versions': {'numpy': np.__version__, 'pandas': pd.__version__, 'librosa': librosa.__version__},
    },
}
with open(SPEC_EXAMPLES_DATA_PATH, 'wb') as f:
    pickle.dump(spec_examples, f, protocol=4)
print(f'\nsaved {SPEC_EXAMPLES_DATA_PATH}  ({os.path.getsize(SPEC_EXAMPLES_DATA_PATH) / 1e6:.1f} MB)')

[TO DO] Fig. 2B column 2: no bird and label in Cell X1 yet
   panel_id        bird label                                                             source
1C_1_regair s_b1349h_23       s_b1349h_23_2024-01-19_10-00-01-01_1729755_1750905_song_regair.wav
1C_1_heliox s_b1349h_23         s_b1349h_23_2024-01-19_12-00-01-01_552590_576145_song_heliox.wav
1D_1_regair z_r11g11_24                                                   bout 183 of 2025-02-17
1D_1_heliox z_r11g11_24                                                   bout 237 of 2025-02-17
2B_1_regair  z_p4g13_23     3                                               bout 1 of 2025-06-27
2B_1_heliox  z_p4g13_23     3                                             bout 134 of 2025-06-27
2B_3_regair z_r11g11_24     5                                               bout 4 of 2025-02-14
2B_3_heliox z_r11g11_24     5                                              bout 84 of 2025-02-14
2D_1_regair s_b1556h_24    22 s_b1556h_24_2024-03-27_12-00-01-02_109

## Fig. 2 and Table 1: syllable features and sound level

A1 measures each syllable's sound level from the recorded audio (it takes 15 to 30 minutes). A2 prints the Table 1 sound level rows. A3 to A6 collect the BioSound features and timing and save them.

In [42]:
# Cell A1: sound level of every syllable, measured from the recorded audio
# Needs Cells 1-4 and F1. Slow (reads every starling wav file and every zebra finch bout): allow 15-30 minutes.
# avg_power cannot serve as loudness: it is the mean of a spectrogram image that was rescaled syllable by
# syllable. This measures level from the audio as recorded: starlings from each song's wav file, zebra
# finches from the unscaled bout waveform in the syllable table (bout_waveform_raw).
#   level_db  RMS of the syllable's samples after a band-pass, in dB relative to full scale (0 dB = a
#             signal filling the whole range of the recording format). Microphone gain differs between
#             birds, so only differences within a bird mean anything.
#   floor_db  the song's background level: the 5th percentile of the levels of its 10 ms frames
#   clipped   the syllable has samples at the top of the recording range, so its true level is higher
from scipy.signal import butter, sosfiltfilt

SYLLABLE_LEVEL_DATA_PATH = _path('HELIOX_SYLLABLE_LEVEL_DATA', os.path.join(DERIVED, 'syllable_level_data.pkl'))
LEVEL_BAND_HZ = (500, 15000)     # level is measured in this band, which leaves out rumble below the song
LEVEL_FILTER_ORDER = 4           # Butterworth, run forwards and backwards (no time shift)
FLOOR_FRAME_MS = 10.0
FLOOR_PERCENTILE = 5
CLIP_AT = 0.999                  # a raw sample at this share of full scale counts as clipped

_band = {}


def first_named(df, name):
    """The column `name` as one Series (one table carries a column name twice)."""
    s = df[name]
    return s.iloc[:, 0] if isinstance(s, pd.DataFrame) else s


def to_full_scale(a):
    """Samples as floats, with 1.0 the largest value the recording format holds."""
    a = np.squeeze(np.asarray(a))
    if a.ndim > 1:
        a = a.mean(axis=1)                                   # channels averaged, as elsewhere here
    if np.issubdtype(a.dtype, np.integer):
        return a.astype(float) / (float(np.iinfo(a.dtype).max) + 1.0)
    return a.astype(float)


def measure_song(raw, rate, on_s, off_s):
    """Per syllable: level (dB FS), peak (share of full scale) and clipped; and the song's floor (dB FS).
    Syllables with unusable times come back as NaN."""
    n_syl = len(on_s)
    level, peak, clipped = np.full(n_syl, np.nan), np.full(n_syl, np.nan), np.zeros(n_syl, dtype=bool)
    x = to_full_scale(raw)
    if x.ndim != 1 or len(x) < int(0.05 * rate):
        return level, peak, clipped, np.nan
    if rate not in _band:
        _band[rate] = butter(LEVEL_FILTER_ORDER, [LEVEL_BAND_HZ[0], min(LEVEL_BAND_HZ[1], 0.45 * rate)],
                             btype='bandpass', fs=rate, output='sos')
    y = sosfiltfilt(_band[rate], x)
    energy = np.concatenate([[0.0], np.cumsum(np.square(y))])     # running sum: mean power of any stretch in one step
    ok = np.isfinite(on_s) & np.isfinite(off_s)
    a = np.clip(np.round(np.where(ok, on_s, 0) * rate).astype(int), 0, len(x))
    b = np.clip(np.round(np.where(ok, off_s, 0) * rate).astype(int), 0, len(x))
    ok &= b > a
    level[ok] = 10 * np.log10((energy[b[ok]] - energy[a[ok]]) / (b[ok] - a[ok]) + 1e-20)
    magnitude = np.abs(x)                                         # peak and clipping come from the raw samples
    for i in np.flatnonzero(ok):
        peak[i] = magnitude[a[i]:b[i]].max()
    clipped[ok] = peak[ok] >= CLIP_AT
    frame = max(1, int(round(FLOOR_FRAME_MS / 1000.0 * rate)))
    edges = np.arange(0, len(x) + 1, frame)
    frame_power = np.diff(energy[edges]) / frame
    floor = 10 * np.log10(np.percentile(frame_power, FLOOR_PERCENTILE) + 1e-20) if len(frame_power) else np.nan
    return level, peak, clipped, floor


def zf_condition(c):
    """The zebra finch condition names, as in Cell E2."""
    s = str(c).lower()
    if 'heliox_fill' in s or 'transition' in s:
        return 'heliox_filling'
    if 'heliox' in s:
        return 'heliox'
    if 'regairb' in s:
        return 'regairb'
    return 'regair' if ('regair' in s or 'regular' in s) else s


frames, problems = [], []

# ---- starlings: the wav file of each song ----------------------------------------------------------
for bird, rel in STARLING_FEATURE_FILES.items():
    df = pd.read_pickle(os.path.join(BASE_DIR, rel))
    song = first_named(df, 'file').map(wav_name)
    t = pd.DataFrame({'species': 'starling', 'bird': bird, 'row': np.arange(len(df)),
                      'condition': first_named(df, 'condition').astype(str).str.lower().to_numpy(),
                      'day': song.str.extract(r'(\d{4}-\d{2}-\d{2})', expand=False).to_numpy(),   # recording date, from the file name
                      'song': song.to_numpy(), 'folder': first_named(df, 'file').map(wav_folder).to_numpy(),
                      'onset_s': pd.to_numeric(first_named(df, 'start_time'), errors='coerce').to_numpy(float),
                      'offset_s': pd.to_numeric(first_named(df, 'end_time'), errors='coerce').to_numpy(float)})
    del df, song
    gc.collect()
    for c in ('level_db', 'peak', 'floor_db'):
        t[c] = np.nan
    t['clipped'] = False
    not_found = 0
    for (folder, name), idx in t.groupby(['folder', 'song'], sort=False).indices.items():
        path = os.path.join(BASE_DIR, folder, name)
        if not os.path.exists(path):
            not_found += 1
            continue
        rate, raw = wavread(path)
        level, peak, clipped, floor = measure_song(raw, rate, t['onset_s'].to_numpy()[idx], t['offset_s'].to_numpy()[idx])
        t.iloc[idx, t.columns.get_loc('level_db')] = level
        t.iloc[idx, t.columns.get_loc('peak')] = peak
        t.iloc[idx, t.columns.get_loc('clipped')] = clipped
        t.iloc[idx, t.columns.get_loc('floor_db')] = floor
    if not_found:
        problems.append(f'{bird}: {not_found} wav files not found under {BASE_DIR}/<folder>/')
    frames.append(t.drop(columns='folder'))
    print(f'  {bird}: {int(t["level_db"].notna().sum())} of {len(t)} syllables measured')

# ---- zebra finches: the unscaled bout waveform in the table -----------------------------------------
for bird, rel in ZF_FEATURE_FILES.items():
    df = pd.read_pickle(os.path.join(BASE_DIR, rel))
    sess = first_named(df, 'sess').astype(str)
    bout = first_named(df, 'bout_index')
    t = pd.DataFrame({'species': 'zebra_finch', 'bird': bird, 'row': np.arange(len(df)),
                      'condition': first_named(df, 'condition').map(zf_condition).to_numpy(),
                      'day': sess.to_numpy(), 'song': (sess + '_bout' + bout.astype(str)).to_numpy(),
                      'onset_s': pd.to_numeric(first_named(df, 'on_ms'), errors='coerce').to_numpy(float) / 1000.0,
                      'offset_s': pd.to_numeric(first_named(df, 'off_ms'), errors='coerce').to_numpy(float) / 1000.0})
    for c in ('level_db', 'peak', 'floor_db'):
        t[c] = np.nan
    t['clipped'] = False
    waves, rates = first_named(df, 'bout_waveform_raw'), first_named(df, 'sample_rate')
    for _, idx in t.groupby('song', sort=False).indices.items():
        raw = waves.iat[idx[0]]                                  # every row of a bout carries the bout's waveform
        if not isinstance(raw, (np.ndarray, list, tuple)):
            continue
        level, peak, clipped, floor = measure_song(raw, float(rates.iat[idx[0]]), t['onset_s'].to_numpy()[idx],
                                                   t['offset_s'].to_numpy()[idx])
        t.iloc[idx, t.columns.get_loc('level_db')] = level
        t.iloc[idx, t.columns.get_loc('peak')] = peak
        t.iloc[idx, t.columns.get_loc('clipped')] = clipped
        t.iloc[idx, t.columns.get_loc('floor_db')] = floor
    del df, waves, rates, sess, bout
    gc.collect()
    frames.append(t)
    print(f'  {bird}: {int(t["level_db"].notna().sum())} of {len(t)} syllables measured')

syllable_levels = pd.concat(frames, ignore_index=True)
del frames
for p in problems:
    print('[CHECK]', p)

# ---- what came out: one line per bird, heliox against regular air ------------------------------------
# The comparison within recording days is the safer one: a day with both conditions has one microphone
# set-up, while whole days of regular air may not.
used = syllable_levels[syllable_levels['condition'].isin(COND_ORDER) & syllable_levels['level_db'].notna()]
for species, s in used.groupby('species', sort=False):
    lines = []
    for bird, b in s.groupby('bird', sort=False):
        med = b.groupby('condition')['level_db'].median()
        floor = b.drop_duplicates(['song', 'condition']).groupby('condition')['floor_db'].median()
        by_day = b.groupby(['day', 'condition'])['level_db'].median().unstack('condition').reindex(columns=COND_ORDER).dropna()
        clip = 100 * b.groupby('condition')['clipped'].mean()
        lines.append(dict(bird=bird, n_regair=int((b['condition'] == 'regair').sum()), n_heliox=int((b['condition'] == 'heliox').sum()),
                          regair_db=med.get('regair', np.nan), heliox_db=med.get('heliox', np.nan),
                          heliox_minus_regair=med.get('heliox', np.nan) - med.get('regair', np.nan),
                          same_day=(by_day['heliox'] - by_day['regair']).mean() if len(by_day) else np.nan,
                          days_both=len(by_day),
                          floor_change=floor.get('heliox', np.nan) - floor.get('regair', np.nan),
                          clipped_regair_pct=clip.get('regair', np.nan), clipped_heliox_pct=clip.get('heliox', np.nan)))
    lines = pd.DataFrame(lines)
    print(f'\n{species}: median syllable level (dB FS) per bird; differences are heliox minus regular air, in dB')
    print(lines.to_string(index=False, float_format=lambda v: f'{v:.1f}'))
    for column, what in [('heliox_minus_regair', 'all days'), ('same_day', 'within days that have both conditions')]:
        d = lines[column].dropna()
        if len(d) > 1:
            print(f'  {what}: mean ± SEM across birds = {d.mean():.2f} ± {d.sem():.2f} dB; '
                  f'lower under heliox in {int((d < 0).sum())} of {len(d)} birds')

require_complete(used, 'syllable levels', 'bird', 'condition', species_col='species')
syllable_level = {
    'syllables': syllable_levels.astype({'species': 'category', 'bird': 'category', 'condition': 'category'}),
    'meta': {'created': datetime.datetime.now().isoformat(timespec='seconds'),
             'source': {'starling': 'the wav file of each song, int16',
                        'zebra_finch': 'bout_waveform_raw in each syllable table, int16'},
             'settings': dict(LEVEL_BAND_HZ=LEVEL_BAND_HZ, LEVEL_FILTER_ORDER=LEVEL_FILTER_ORDER,
                              FLOOR_FRAME_MS=FLOOR_FRAME_MS, FLOOR_PERCENTILE=FLOOR_PERCENTILE, CLIP_AT=CLIP_AT),
             'notes': {'row': "position of the syllable in its bird's syllable table (0 = first row), so these values "
                              'can be attached to that table by position',
                       'level_db': 'RMS level of the band-passed syllable, dB re full scale',
                       'floor_db': "background level of the syllable's song: a low percentile of its frame levels",
                       'peak, clipped': 'from the raw samples, before the band-pass',
                       'condition': 'regair, heliox, and the conditions the analyses leave out (regairb, heliox_filling)'},
             'versions': {'numpy': np.__version__, 'pandas': pd.__version__}},
}
with open(SYLLABLE_LEVEL_DATA_PATH, 'wb') as f:
    pickle.dump(syllable_level, f, protocol=4)
print(f'\nsaved {SYLLABLE_LEVEL_DATA_PATH}  ({os.path.getsize(SYLLABLE_LEVEL_DATA_PATH) / 1e6:.1f} MB)')

  s_b1349h_23: 12683 of 12683 syllables measured
  s_b1354_25: 20051 of 20051 syllables measured
  s_b1364h_25: 50579 of 50579 syllables measured
  s_b1481h_24: 2336 of 2336 syllables measured
  s_b1484h_24: 6928 of 6928 syllables measured
  s_b1556h_24: 6023 of 6023 syllables measured
  z_r11g11_24: 6273 of 6273 syllables measured
  z_p7y2_23: 2859 of 2859 syllables measured
  z_y11c13_24: 17118 of 17118 syllables measured
  z_c8c19_24: 9174 of 9174 syllables measured
  z_p4g13_23: 9342 of 9342 syllables measured
  z_g14g16_24: 18962 of 18962 syllables measured

starling: median syllable level (dB FS) per bird; differences are heliox minus regular air, in dB
       bird  n_regair  n_heliox  regair_db  heliox_db  heliox_minus_regair  same_day  days_both  floor_change  clipped_regair_pct  clipped_heliox_pct
s_b1349h_23      4451      8232      -53.7      -57.1                 -3.4      -5.2          3           0.6                 0.0                 0.0
 s_b1354_25     15100      2238 

In [43]:
# Cell A2: heliox effect on syllable level (the numbers that replace "Average power" in Table 1)
# Stands alone: reads syllable_level_data.pkl from Cell A1 and nothing else.
# Every effect is Regair minus Heliox in dB, as in Table 1, so a POSITIVE value means quieter under heliox.
import os, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

LEVEL_PATH = os.environ.get(
    'HELIOX_SYLLABLE_LEVEL_DATA',
    '/mnt/cube/lstanwic/umap_avgn_shared/lauren_umap/heliox_processing/data/derived/syllable_level_data.pkl')

lv = pd.read_pickle(LEVEL_PATH)['syllables']
lv = lv[lv['condition'].isin(['regair', 'heliox']) & lv['level_db'].notna()].copy()
for c in ('species', 'bird', 'condition', 'day'):
    lv[c] = lv[c].astype(str)
lv['regair'] = (lv['condition'] == 'regair').astype(int)       # coefficient of regair = Regair minus Heliox
# A day with both conditions has one microphone set-up; whole days of regular air may not.
lv['day_has_both'] = lv.groupby(['bird', 'day'])['condition'].transform('nunique') == 2


def condition_effect(d):
    """level_db ~ condition + (1 | bird) on syllables, the model behind every Table 1 row."""
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')                          # a bird variance of zero is reported below instead
        res = smf.mixedlm('level_db ~ regair', d, groups=d['bird']).fit()
    lo, hi = res.conf_int().loc['regair']
    z = res.tvalues['regair']
    log10_p = (np.log(2) + stats.norm.logsf(abs(z))) / np.log(10)   # exact where 2 * sf(|z|) would round to 0
    return dict(beta=res.params['regair'], SE=res.bse['regair'], z=z, P=10 ** log10_p, log10_P=log10_p,
                CI_low=lo, CI_high=hi, syllables=len(d), birds=d['bird'].nunique(),
                bird_variance=float(res.cov_re.iloc[0, 0]), converged=bool(res.converged))


def bird_differences(d, within_day):
    """Each bird's Regair minus Heliox difference in median level: over all its syllables, or as the mean
    of the differences on the days that have both conditions."""
    if within_day:
        m = d[d['day_has_both']].groupby(['bird', 'day', 'condition'])['level_db'].median().unstack('condition')
        return (m['regair'] - m['heliox']).groupby('bird').mean()
    m = d.groupby(['bird', 'condition'])['level_db'].median().unstack('condition')
    return m['regair'] - m['heliox']


model_rows, bird_rows = [], []
for species in ['zebra_finch', 'starling']:                      # the column order of Table 1
    d = lv[lv['species'] == species]
    for data, sub in [('all days', d), ('days with both conditions', d[d['day_has_both']]),
                      ('all days, clipped syllables left out', d[~d['clipped'].astype(bool)])]:
        if sub['bird'].nunique() > 1 and sub['regair'].nunique() == 2:
            model_rows.append(dict(species=species, data=data, **condition_effect(sub)))
    for data, within_day in [('all days', False), ('days with both conditions', True)]:
        diff = bird_differences(d, within_day).dropna()
        t, p = stats.ttest_1samp(diff, 0.0) if len(diff) > 1 else (np.nan, np.nan)
        bird_rows.append(dict(species=species, data=data, birds=len(diff), mean=diff.mean(), SEM=diff.sem(), t=t,
                              df=len(diff) - 1, P=p, quieter_in_heliox=f'{int((diff > 0).sum())}/{len(diff)}',
                              per_bird=', '.join(f'{v:.1f}' for v in diff)))

models, by_bird = pd.DataFrame(model_rows), pd.DataFrame(bird_rows)
print('Syllable-level models, level_db ~ condition + (1 | bird); beta = Regair minus Heliox (dB)')
print(models.drop(columns='log10_P').to_string(index=False, float_format=lambda v: f'{v:.4g}'))
print('\nBird-level: each bird\'s Regair minus Heliox difference in median level (dB), one-sample t-test across birds')
with pd.option_context('display.width', 250, 'display.max_colwidth', 80):
    print(by_bird.to_string(index=False, float_format=lambda v: f'{v:.4g}'))
print('\nP below 1e-300, as log10:')
print(models.loc[models['P'] == 0, ['species', 'data', 'log10_P']].to_string(index=False) if (models['P'] == 0).any() else '  none')
print('\nclipped syllables (%), by species and condition:')
print((100 * lv.groupby(['species', 'condition'])['clipped'].mean()).unstack('condition').round(2).to_string())

Syllable-level models, level_db ~ condition + (1 | bird); beta = Regair minus Heliox (dB)
    species                                 data   beta      SE     z          P  CI_low  CI_high  syllables  birds  bird_variance  converged
zebra_finch                             all days 0.7487 0.04338 17.26  9.317e-67  0.6637   0.8337      63728      6          8.527       True
zebra_finch            days with both conditions 0.5555 0.04671 11.89   1.29e-32   0.464   0.6471      57883      6          9.223       True
zebra_finch all days, clipped syllables left out 0.7173 0.04316 16.62   5.12e-62  0.6327   0.8019      62777      6          7.815       True
   starling                             all days  2.772 0.07311 37.91 1.464e-314   2.629    2.915      95063      6          51.83       True
   starling            days with both conditions  3.908  0.1292 30.24 7.608e-201   3.655    4.161      25385      5           63.9       True
   starling all days, clipped syllables left out  2.772 0.

In [44]:
# Cell A3: syllable features (Fig. 2 A, C, G-J and Supp. Fig. S2), settings
# Feeds syllable_features_figures.ipynb. Run after Cells 1-4 and Cell F1.
import gc

SYLLABLE_FEATURES_DATA_PATH = _path('HELIOX_SYLLABLE_FEATURES_DATA',
                                    os.path.join(DERIVED, 'syllable_features_data.pkl'))
PAF_CACHE_DIR = os.path.join(DERIVED, 'biosound_paf_slim')
os.makedirs(PAF_CACHE_DIR, exist_ok=True)

# One BioSound table per bird: the same files as Cell E3, repeated so these cells run without it.
PAF_FILES = {
    'starling': {
        's_b1349h_23': 's_b1349h_23/s-b1349-biosound-sylldf-2025-nonr.pickle',
        's_b1354_25':  's_b1354_25/s-b1354-biosound-sylldf-2025.pickle',
        's_b1364h_25': 's_b1364h_25/s-b1364-biosound-sylldf-2025.pickle',
        's_b1481h_24': 's_b1481h_24/biosound-features-biosound-df-2025.pickle',
        's_b1484h_24': 's_b1484h_24/s-b1484-biosound-sylldf-2025.pickle',
        's_b1556h_24': 's_b1556h_24/s-b1556-biosound-sylldf-2025.pickle'},
    'zebra_finch': {
        'z_c8c19_24':  'z_c8c19_24/biosound-features-biosound-df-2025.pickle',
        'z_g14g16_24': 'z_g14g16_24/biosound-features-biosound-df-2025.pickle',
        'z_p4g13_23':  'z_p4g13_23/biosound-features-biosound-df-2025.pickle',
        'z_p7y2_23':   'z_p7y2_23/biosound-features-biosound-df-2025.pickle',
        'z_r11g11_24': 'z_r11g11_24/r11g11-biosound-sylldf-2025.pickle',
        'z_y11c13_24': 'z_y11c13_24/y11c13-biosound-sylldf-2025.pickle'},
}

# The 18 BioSound predefined acoustic features (PAFs) of Supp. Fig. S2, by their column names:
# 8 spectral, 5 temporal, then pitch saliency, mean / min / max F0 and the first formant.
PAF_COLUMNS = ['meanS', 'stdS', 'skewS', 'kurtS', 'entS', 'q1', 'q2', 'q3',
               'meanT', 'stdT', 'skewT', 'kurtT', 'entT',
               'sal', 'fund', 'minfund', 'maxfund', 'F1']
# Fig. 2 H-J, in panel order. level_db, the syllable's sound level measured from the recorded audio (Cell A1),
# replaces avg_power: that was the mean of each syllable's spectrogram image, and the images were rescaled
# syllable by syllable, so it could not follow loudness. Fig. 2 G is BioSound spectral entropy (entS, Cell A4);
# the old spectral_entropy column was the entropy of the spectrogram averaged over frequency, so it measured how
# energy is spread over time, and it is no longer used.
TIMING_FEATURES = ['level_db', 'duration', 'isi']
SYLLABLE_LEVEL_DATA_PATH = _path('HELIOX_SYLLABLE_LEVEL_DATA', os.path.join(DERIVED, 'syllable_level_data.pkl'))

# Syllables per condition (regair, heliox) that these tables gave when the original notebooks ran, read
# off their saved output. Cells A4 and A5 say so if the tables on disk no longer give these counts.
PAF_SYLLABLES_ORIGINAL = {'starling': (76843, 21726), 'zebra_finch': (41417, 26975)}
TIMING_SYLLABLES_ORIGINAL = {'starling': (73335, 21728), 'zebra_finch': (37130, 26598)}


def count_line(df, original):
    """Syllables per condition, with a flag when they are not the counts of the original analysis."""
    n = tuple(int((df['condition'] == c).sum()) for c in COND_ORDER)
    same = '' if n == tuple(original) else f'   [CHECK] the original analysis had {original[0]} / {original[1]}'
    return f'{len(df)} syllables ({n[0]} regair / {n[1]} heliox){same}'


print('syllable features ->', SYLLABLE_FEATURES_DATA_PATH)

syllable features -> /mnt/cube/lstanwic/umap_avgn_shared/lauren_umap/heliox_processing/data/derived/syllable_features_data.pkl


In [45]:
# Cell A4: the 18 BioSound features of every syllable (Fig. 2 A, C and Supp. Fig. S2)
# The source tables run to ~10 GB each, so each bird is cut down once to the columns needed here and
# kept in biosound_paf_slim/. Later runs read only those small files. (Cell E3 keeps its own copies with
# three features; these hold all 18, and the condition label as it was recorded.)
# The recorded label is also saved, in the column 'label': Fig. 2 G fits spectral entropy (entS) in the same
# model as amplitude, duration and ISI, which use only syllables labelled regair or heliox.
def load_paf(bird, rel):
    slim_path = os.path.join(PAF_CACHE_DIR, f'{bird}_paf.pickle')
    if not os.path.exists(slim_path):
        slim = read_columns(os.path.join(BASE_DIR, rel), PAF_COLUMNS + ['condition'])
        absent = [c for c in PAF_COLUMNS + ['condition'] if c not in slim.columns]
        if absent:
            raise KeyError(f'{bird}: no column(s) {absent} in {rel}')
        for f in PAF_COLUMNS:
            # float32 keeps seven significant digits and halves the file
            slim[f] = pd.to_numeric(slim[f], errors='coerce').astype('float32')
        slim['condition'] = slim['condition'].astype(str)
        slim[['condition'] + PAF_COLUMNS].reset_index(drop=True).to_pickle(slim_path)
    return pd.read_pickle(slim_path)


a_biosound, a_labels = {}, {}
for species, files in PAF_FILES.items():
    frames = []
    for bird, rel in files.items():
        one = load_paf(bird, rel)
        one.insert(0, 'bird', bird)
        frames.append(one)
    df = pd.concat(frames, ignore_index=True)
    del frames
    gc.collect()
    recorded = df['condition'].str.lower()
    # The BioSound analysis in the paper counts a syllable as heliox when its label contains 'heliox'
    # and as regular air otherwise. Kept as it is, so that the models reproduce the paper; the table printed
    # below shows which recorded labels end up in each condition.
    df['condition'] = np.where(recorded.str.contains('heliox'), 'heliox', 'regair')
    df['label'] = recorded
    a_labels[species] = pd.crosstab(recorded.rename('label as recorded'), df['condition'].rename('counted as'))
    a_biosound[species] = df[['bird', 'condition', 'label'] + PAF_COLUMNS]

    print(f'\n{species}: {count_line(df, PAF_SYLLABLES_ORIGINAL[species])}')
    print(df.groupby(['bird', 'condition']).size().unstack(fill_value=0).to_string())
    print(a_labels[species].to_string())
    other = [label for label in a_labels[species].index if label not in COND_ORDER]
    if other:
        print(f'  [CHECK] syllables labelled {other} are counted in the conditions shown above. If any of these '
              'is a chamber-filling or post-heliox period, Fig. 2 A, C and Supp. Fig. S2 include it.')
    empty = df[PAF_COLUMNS].isna().sum()
    if empty.any():
        print('  syllables with no value, left out of that feature only:', empty[empty > 0].to_dict())
del df
gc.collect()


starling: 98569 syllables (76843 regair / 21726 heliox)
condition    heliox  regair
bird                       
s_b1349h_23    8232    4451
s_b1354_25     2238   17808
s_b1364h_25    6827   43751
s_b1481h_24     695    1632
s_b1484h_24    2502    4416
s_b1556h_24    1232    4785
counted as         heliox  regair
label as recorded                
heliox              21726       0
regair                  0   73306
regairb                 0    3537
  [CHECK] syllables labelled ['regairb'] are counted in the conditions shown above. If any of these is a chamber-filling or post-heliox period, Fig. 2 A, C and Supp. Fig. S2 include it.

zebra_finch: 68392 syllables (41417 regair / 26975 heliox)
condition    heliox  regair
bird                       
z_c8c19_24     5125    5060
z_g14g16_24    8762   11194
z_p4g13_23     3915    6618
z_p7y2_23       415    2279
z_r11g11_24    2511    5409
z_y11c13_24    6247   10857
counted as         heliox  regair
label as recorded                
heliox     

35

In [46]:
# Cell A5: duration, ISI and sound level of every syllable (Fig. 2 H-J)
# Duration and ISI come from one syllable table per bird (the tables of Cell F1), as in the
# original analysis. Sound level comes from syllable_level_data.pkl (Cell A1), whose `row` is the syllable's
# position in that same table, so the two are joined on bird and row.
TABLE_FEATURES = ['duration', 'isi']


def bird_key(bird):
    """Starlings by number, since bird 1364 goes by two names; zebra finches by name."""
    return bird_num(bird) if str(bird).startswith('s_') else str(bird)


level = pd.read_pickle(SYLLABLE_LEVEL_DATA_PATH)
if isinstance(level, dict):          # in case the level table was saved inside a dict
    level = next(v for v in level.values() if isinstance(v, pd.DataFrame) and 'level_db' in v.columns)
absent = {'bird', 'row', 'condition', 'level_db', 'clipped'} - set(level.columns)
if absent:
    raise KeyError(f'syllable_level_data.pkl has no column(s) {sorted(absent)}')
level = pd.DataFrame({'key': level['bird'].map(bird_key).to_numpy(), 'row': level['row'].astype(int).to_numpy(),
                      'level_condition': level['condition'].astype(str).str.lower().to_numpy(),
                      'level_db': pd.to_numeric(level['level_db'], errors='coerce').to_numpy(float),
                      'clipped': level['clipped'].astype(bool).to_numpy()})

a_timing, used = {}, 0
for species, files in [('starling', STARLING_FEATURE_FILES), ('zebra_finch', ZF_FEATURE_FILES)]:
    frames, left_out = [], {}
    for bird, rel in files.items():
        df = read_columns(os.path.join(BASE_DIR, rel), ['condition', 'isi', 'time_between_syllables'] + TABLE_FEATURES)
        if 'isi' not in df.columns and 'time_between_syllables' in df.columns:
            df['isi'] = df['time_between_syllables']          # the starling tables' name for the ISI
        absent = [c for c in ['condition'] + TABLE_FEATURES if c not in df.columns]
        if absent:
            raise KeyError(f'{bird}: no column(s) {absent} in {rel}')
        # row: position in the full table, before anything is left out, which is what the level table counts
        one = pd.DataFrame({'bird': bird, 'row': np.arange(len(df)),
                            'condition': df['condition'].astype(str).str.lower().to_numpy()})
        for f in TABLE_FEATURES:
            one[f] = pd.to_numeric(df[f], errors='coerce').replace([np.inf, -np.inf], np.nan).to_numpy(float)
        # regular air and heliox only: chamber-filling periods and regular air recorded after heliox are
        # not part of this analysis
        other = one.loc[~one['condition'].isin(COND_ORDER), 'condition'].value_counts()
        for label, n in other.items():
            left_out[label] = left_out.get(label, 0) + int(n)
        frames.append(one[one['condition'].isin(COND_ORDER)])
        del df, one
        gc.collect()
    t = pd.concat(frames, ignore_index=True)
    del frames

    # sound level of each syllable: every syllable must find its level, in the same condition
    t = t.assign(key=t['bird'].map(bird_key)).merge(level, on=['key', 'row'], how='left', validate='one_to_one',
                                                    indicator=True)
    unmatched = int((t['_merge'] != 'both').sum())
    disagree = int(((t['_merge'] == 'both') & (t['level_condition'] != t['condition'])).sum())
    if unmatched or disagree:
        raise ValueError(f'{species}: {unmatched} syllables have no row in syllable_level_data.pkl and {disagree} '
                         'have a different condition there, so bird and row do not identify the same syllables')
    used += len(t)
    a_timing[species] = t[['bird', 'condition'] + TIMING_FEATURES + ['clipped']]

    t = a_timing[species]
    if not (0.01 < t['duration'].median() < 5):
        raise ValueError(f'{species}: median duration is {t["duration"].median():.4g}; the figures expect seconds')
    print(f'\n{species}: {count_line(t, TIMING_SYLLABLES_ORIGINAL[species])}'
          + (f'   [left out: {left_out}]' if left_out else ''))
    print(t.groupby(['bird', 'condition']).size().unstack(fill_value=0)
          .assign(clipped=t.groupby('bird')['clipped'].sum()).to_string())
    print('  medians:  ' + ',  '.join(f'{f} {t[f].median():.4g}' for f in TIMING_FEATURES)
          + f';  syllables with no ISI: {int(t["isi"].isna().sum())}')
print(f'\nsound level: {used} of the {len(level)} syllables in syllable_level_data.pkl joined'
      + ('' if used == len(level) else '   [CHECK] the rest belong to no syllable here'))


starling: 95063 syllables (73335 regair / 21728 heliox)   [left out: {'regairb': 3537}]
condition    heliox  regair  clipped
bird                                
s_b1349h_23    8232    4451        0
s_b1354_25     2238   15100        0
s_b1364h_25    6827   42928        0
s_b1481h_24     696    1640        0
s_b1484h_24    2502    4426        0
s_b1556h_24    1233    4790        0
  medians:  level_db -53.88,  duration 0.444,  isi 0.088;  syllables with no ISI: 2679

zebra_finch: 63728 syllables (37130 regair / 26598 heliox)
condition    heliox  regair  clipped
bird                                
z_c8c19_24     5011    4163        3
z_g14g16_24    8764   10198        0
z_p4g13_23     3871    5471        0
z_p7y2_23       466    2393       66
z_r11g11_24    2232    4041      211
z_y11c13_24    6254   10864      671
  medians:  level_db -25.26,  duration 0.119,  isi 0.04;  syllables with no ISI: 2027

sound level: 158791 of the 162328 syllables in syllable_level_data.pkl joined   [CHEC

In [47]:
# Cell A6: check and save the syllable-feature data
syllable_features = {
    'biosound_starling': a_biosound['starling'], 'biosound_zebra_finch': a_biosound['zebra_finch'],
    'timing_starling': a_timing['starling'], 'timing_zebra_finch': a_timing['zebra_finch'],
}
for name, df in syllable_features.items():
    # all six birds of the species, in both conditions, with no cluster paths in the table
    require_complete(df, name, 'bird', 'condition',
                     species='zebra_finch' if name.endswith('zebra_finch') else 'starling')
    assert_portable(name, df)

syllable_features['meta'] = {
    'created': datetime.datetime.now().isoformat(timespec='seconds'),
    'biosound_features': PAF_COLUMNS,
    'timing_features': TIMING_FEATURES,
    'tables': {'biosound': {sp: {b: os.path.basename(rel) for b, rel in files.items()}
                            for sp, files in PAF_FILES.items()},
               'timing': {'starling': {b: os.path.basename(rel) for b, rel in STARLING_FEATURE_FILES.items()},
                          'zebra_finch': {b: os.path.basename(rel) for b, rel in ZF_FEATURE_FILES.items()}},
               'level': os.path.basename(SYLLABLE_LEVEL_DATA_PATH)},
    'condition': {'biosound': "heliox if the recorded label contains 'heliox', regair otherwise",
                  'timing': 'syllables labelled regair or heliox; every other label left out',
                  'biosound label': 'the label as recorded; Fig. 2 G keeps the syllables labelled regair or heliox',
                  # recorded label -> how many syllables were counted in each condition
                  'biosound_labels': {sp: {str(label): {c: int(n) for c, n in row.items()}
                                           for label, row in tab.iterrows()} for sp, tab in a_labels.items()}},
    'units': {'duration, isi': 's', 'meanS, stdS, q1, q2, q3, fund, minfund, maxfund, F1': 'Hz', 'meanT, stdT': 's',
              'level_db': 'dB re full scale: RMS level in the 500 Hz to 15 kHz band of the recorded audio',
              'the other features': 'no unit'},
    'notes': {'biosound': 'BioSound predefined acoustic features, one row per syllable, stored as float32',
              'timing': 'one row per syllable; syllables without an ISI are kept; nothing has been trimmed '
                        'here (the figure notebook drops implausible durations and ISIs)',
              'level_db': 'from syllable_level_data.pkl (Cell A1), joined on bird and row of the syllable table',
              'clipped': 'syllables flagged as clipped in syllable_level_data.pkl; kept in the tables',
              's_b1364h_25': 'all syllables of this bird (the table that is not balanced across conditions)'},
    'versions': {'numpy': np.__version__, 'pandas': pd.__version__},
}
with open(SYLLABLE_FEATURES_DATA_PATH, 'wb') as f:
    pickle.dump(syllable_features, f, protocol=4)
print(f'\nsaved {SYLLABLE_FEATURES_DATA_PATH}  ({os.path.getsize(SYLLABLE_FEATURES_DATA_PATH) / 1e6:.1f} MB)')

biosound_starling: complete (98569 rows)
biosound_zebra_finch: complete (68392 rows)
timing_starling: complete (95063 rows)
timing_zebra_finch: complete (63728 rows)

saved /mnt/cube/lstanwic/umap_avgn_shared/lauren_umap/heliox_processing/data/derived/syllable_features_data.pkl  (22.8 MB)


## Fig. 3: transition structure

In [48]:
# Cell T1: syllable transition structure (Fig. 3), for transition_structure_figures.ipynb
# Needs Cells 1-4 and F1. Four tables: every transition of every bird (Fig. 3B, D) and the
# syllables of the two example birds whose transitions are drawn as diagrams (Fig. 3A, C).
TRANSITION_DATA_PATH = _path('HELIOX_TRANSITION_DATA', os.path.join(DERIVED, 'transition_structure_data.pkl'))

# Example birds.
ZF_EXAMPLE = 'z_g14g16_24'
STARLING_EXAMPLE = dict(bird='s_b1349h_23', label='hdbscan_labels',
                        table=os.path.join(BASE_DIR, 's_b1349h_23', 'syllable_df_labels_mindist2_20250109.pickle'))
# Zebra finch songs left out of the transition counts in the original analysis: every song of that
# bird holding one of these syllable types.
ZF_DROP_SONGS_WITH = {'p7y2': [4], 'y11c13': [7]}


def zf_name(indv):
    """'p7y2' or 'z_p7y2_23' -> 'z_p7y2_23' (the combined zebra finch table uses both forms)."""
    if indv in ZF_FEATURE_FILES:
        return indv
    hits = [k for k in ZF_FEATURE_FILES if k.split('_')[1] == indv]
    if len(hits) != 1:
        raise KeyError(f'zebra finch {indv!r} matches {hits} in ZF_FEATURE_FILES')
    return hits[0]


# ---- 1. zebra finches: every transition (Fig. 3B) --------------------------------------------------
zf = read_columns(ZF_SYL_PATH, ['indv', 'song_id', 'syllable_index', 'label', 'isi', 'condition'])
zf['indv'] = zf['indv'].astype(str)
for short, bad in ZF_DROP_SONGS_WITH.items():
    songs = zf.loc[(zf['indv'] == short) & zf['label'].isin(bad), 'song_id'].unique()
    print(f'{short}: {len(songs)} songs with syllable type(s) {bad} left out')
    zf = zf[~((zf['indv'] == short) & zf['song_id'].isin(songs))]
zf = zf.sort_values(['indv', 'song_id', 'syllable_index'], kind='mergesort')
zf['from'] = zf.groupby(['indv', 'song_id'])['label'].shift(1)    # the syllable before, within the song
# a transition needs a syllable before it and a measured gap ('isi', s, is the gap before this syllable)
zf = zf.dropna(subset=['from', 'isi'])
zf['condition'] = zf['condition'].astype(str).str.lower()
print('zebra finch conditions left out of the transition table:',
      zf.loc[~zf['condition'].isin(COND_ORDER), 'condition'].value_counts().to_dict() or 'none')
zf = zf[zf['condition'].isin(COND_ORDER)]
zf_transitions = pd.DataFrame({'bird': zf['indv'].map(zf_name).to_numpy(), 'condition': zf['condition'].to_numpy(),
                               'from_label': zf['from'].astype(int).to_numpy(), 'to_label': zf['label'].astype(int).to_numpy(),
                               'isi_s': zf['isi'].astype(float).to_numpy()})
del zf
gc.collect()

# ---- 2. starlings: every transition (Fig. 3B, D) ----------------------------------------------------
# As in the original analysis: unclustered syllables (-1) are kept as a syllable type of their own,
# a bout is one wav file, and its condition is that of its first syllable.
frames = []
for bird in STARLING_BIRDS:
    df = read_columns(os.path.join(AUGMENTED_DIR, bird, f'{bird}_syllable_df_augmented.pkl'),
                      ['file', BOUTCOL, 'condition', 'onsets_ms', 'offsets_ms', LABCOL])
    d = pd.DataFrame({
        'condition': df['condition'].astype(str).str.lower().to_numpy(),
        'bout': df[BOUTCOL if BOUTCOL in df.columns else 'file'].map(str).to_numpy(),
        'onset_s': pd.to_numeric(df['onsets_ms'], errors='coerce').to_numpy(),    # seconds, despite the name
        'offset_s': pd.to_numeric(df['offsets_ms'], errors='coerce').to_numpy(),
        'label': pd.to_numeric(df[LABCOL], errors='coerce').to_numpy()}).sort_values(['bout', 'onset_s'], kind='mergesort')
    del df
    if d['label'].isna().any():
        raise ValueError(f'{bird}: {int(d["label"].isna().sum())} syllables have no {LABCOL}')
    mixed = int((d.groupby('bout')['condition'].nunique() > 1).sum())
    if mixed:
        print(f'  [note] {bird}: {mixed} bouts hold more than one condition; each takes its first syllable\'s')
    bout = d['bout'].to_numpy()
    lab, on, off = d['label'].to_numpy(int), d['onset_s'].to_numpy(float), d['offset_s'].to_numpy(float)
    first_cond = d.groupby('bout', sort=False)['condition'].transform('first').to_numpy()
    i = np.flatnonzero(bout[1:] == bout[:-1])     # syllable i and the next one in the same bout
    frames.append(pd.DataFrame({'bird': bird, 'condition': first_cond[i], 'from_label': lab[i], 'to_label': lab[i + 1],
                                'isi_s': on[i + 1] - off[i]}))
starling_transitions = pd.concat(frames, ignore_index=True)
del frames, d
gc.collect()
other = sorted(set(starling_transitions['condition']) - set(COND_ORDER))
if other:
    raise ValueError(f'conditions other than regair and heliox in the starling tables: {other}')

# ---- 3. the zebra finch drawn in Fig. 3A --------------------------------------------------------------
df = read_columns(os.path.join(BASE_DIR, ZF_FEATURE_FILES[ZF_EXAMPLE]), ['condition', 'sess', 'bout_index', 'on_ms', 'label'])
df['condition'] = df['condition'].astype(str).str.lower()
df = df[df['condition'].isin(COND_ORDER)].sort_values(['sess', 'bout_index', 'on_ms'], kind='mergesort')
zf_example = pd.DataFrame({'bird': ZF_EXAMPLE, 'condition': df['condition'].to_numpy(),
                           # bout numbers start again each day, so a song is a day plus a bout number
                           'song': (df['sess'].astype(str) + '|' + df['bout_index'].astype(str)).to_numpy(),
                           'label': df['label'].astype(int).to_numpy()})

# ---- 4. the starling drawn in Fig. 3C ---------------------------------------------------------------
lab_col = STARLING_EXAMPLE['label']
df = read_columns(STARLING_EXAMPLE['table'], ['condition', 'syllables_sequence_id', 'syllables_sequence_pos',
                                              'umap', 'umap_x', 'umap_y', lab_col])
df['condition'] = df['condition'].astype(str).str.lower()
x, y = umap_xy(df)
starling_example = pd.DataFrame({'bird': STARLING_EXAMPLE['bird'], 'condition': df['condition'].to_numpy(),
                                 'song': df['syllables_sequence_id'].to_numpy(),
                                 'position': df['syllables_sequence_pos'].to_numpy(),
                                 'label': pd.to_numeric(df[lab_col]).astype(int).to_numpy(), 'umap_x': x, 'umap_y': y})
starling_example = starling_example[starling_example['condition'].isin(COND_ORDER)].reset_index(drop=True)
# The original diagrams took syllables in table order; say so if that is not position order anywhere.
out_of_order = int((starling_example.groupby('song')['position'].apply(lambda p: (np.diff(p.to_numpy()) < 0).any())).sum())
n_types = starling_example.loc[starling_example['label'] >= 0, 'label'].nunique()
print(f'{STARLING_EXAMPLE["bird"]} example: {n_types} syllable types, {starling_example["song"].nunique()} songs'
      + (f'   [note] {out_of_order} songs are not in position order in the table; the figure sorts by position' if out_of_order else ''))
del df
gc.collect()

transition_structure = {'zf_transitions': zf_transitions, 'starling_transitions': starling_transitions,
                        'zf_example': zf_example, 'starling_example': starling_example}
for name, df in transition_structure.items():
    if name.endswith('transitions'):
        # all six birds of the species in both conditions
        require_complete(df, name, 'bird', 'condition', species='zebra_finch' if name.startswith('zf') else 'starling')
    assert_portable(name, df)
for name in ('zf_transitions', 'starling_transitions'):
    t = transition_structure[name]
    print(f'\n{name}:')
    print(t.groupby(['bird', 'condition']).size().unstack(fill_value=0).to_string())

transition_structure['meta'] = {
    'created': datetime.datetime.now().isoformat(timespec='seconds'),
    'sources': {'zf_transitions': os.path.basename(ZF_SYL_PATH),
                'starling_transitions': 'one <bird>_syllable_df_augmented.pkl per starling, as in Cell W2',
                'zf_example': os.path.basename(ZF_FEATURE_FILES[ZF_EXAMPLE]),
                'starling_example': os.path.basename(STARLING_EXAMPLE['table'])},
    'settings': dict(LABEL_COLUMN=LABCOL, ZF_DROP_SONGS_WITH=ZF_DROP_SONGS_WITH, ZF_EXAMPLE=ZF_EXAMPLE,
                     STARLING_EXAMPLE_LABEL=lab_col),
    'notes': {'transitions': 'neighbouring syllables of one song; isi_s = gap between them (s)',
              'starling_transitions': 'unclustered syllables kept as type -1, as in the original analysis',
              'examples': 'one row per syllable, in song order; umap_x, umap_y place the starling diagram nodes'},
    'versions': {'numpy': np.__version__, 'pandas': pd.__version__},
}
with open(TRANSITION_DATA_PATH, 'wb') as f:
    pickle.dump(transition_structure, f, protocol=4)
print(f'\nsaved {TRANSITION_DATA_PATH}  ({os.path.getsize(TRANSITION_DATA_PATH) / 1e6:.1f} MB)')

p7y2: 44 songs with syllable type(s) [4] left out
y11c13: 35 songs with syllable type(s) [7] left out
zebra finch conditions left out of the transition table: none
s_b1349h_23 example: 22 syllable types, 608 songs
zf_transitions: complete (58661 rows)
starling_transitions: complete (60038 rows)

zf_transitions:
condition    heliox  regair
bird                       
z_c8c19_24     4874    4060
z_g14g16_24    8480    9950
z_p4g13_23     3613    5208
z_p7y2_23       368    1192
z_r11g11_24    2151    3898
z_y11c13_24    5796    9071

starling_transitions:
condition    heliox  regair
bird                       
s_b1349h_23    7843    4232
s_b1354_25     2157   17191
s_b1364_25     6524    7251
s_b1481h_24     641    1564
s_b1484h_24    2422    4307
s_b1556h_24    1207    4699

saved /mnt/cube/lstanwic/umap_avgn_shared/lauren_umap/heliox_processing/data/derived/transition_structure_data.pkl  (5.4 MB)


## Fig. 4: song structure and recurring k-grams

In [49]:
# Cell SS1: song structure (Fig. 4 C-G) for song_structure_figures.ipynb
# Zebra finch: every syllable (intro note or not, duration, ISI) and the motifs in every song.
# Starling: every syllable with its position in the bout (duration, ISI).
# Run after Cells 1-3, F1 (read_columns, STARLING_FEATURE_FILES) and 4 (assert_portable).
from collections import Counter, defaultdict

SONG_STRUCTURE_DATA_PATH = _path('HELIOX_SONG_STRUCTURE_DATA', os.path.join(DERIVED, 'song_structure_data.pkl'))
INTRO_LABEL = 0      # zebra finch intro notes were given label 0 when the clusters were named
# the combined zebra finch table names three birds without the prefix used everywhere else
ZF_FULL_NAME = {'p7y2': 'z_p7y2_23', 'r11g11': 'z_r11g11_24', 'y11c13': 'z_y11c13_24'}


def motif_order(labels):
    """A bird's motif, found as in the original analysis: start at the syllable type with the most
    transitions out of it (a type repeating itself is not counted), then keep following the most
    common next type until the sequence returns to the start. `labels` is the bird's whole table
    in row order, so transitions across song boundaries count too, as they did originally."""
    pairs = Counter((a, b) for a, b in zip(labels[:-1], labels[1:]) if a != b)
    out = defaultdict(int)
    for (a, _), n in pairs.items():
        out[a] += n
    start = max(out, key=out.get)
    order, current = [start], start
    while True:
        following = sorted(((b, n) for (a, b), n in pairs.items() if a == current), key=lambda x: -x[1])
        step = None
        for b, _ in following:
            if b == start:
                return order                 # back at the start: the motif is complete
            if b not in order:
                step = b
                break
        if step is None:
            return order                     # dead end before the loop closed
        order.append(step)
        current = step


def motifs_in_song(labels, first, length):
    """Motifs in one song, counted as in the paper (the 'spacing' count): appearances of the motif's
    first syllable that come at least one motif length after the last counted one. The song's first
    appearance starts the count and is not itself counted."""
    starts = [i for i, lab in enumerate(labels) if lab == first]
    n = 0
    last = starts[0] if starts else None
    for i in starts[1:]:
        if i - last >= length:
            n += 1
            last = i
    return n


# ---- zebra finch: the combined table of all six birds, kept in its row order ------------------
# Row order matters: the motif is inferred from the whole table in order, and each song's labels
# are read in order.
zf = read_columns(ZF_SYL_PATH, ['indv', 'song_id', 'heliox', 'condition', 'label', 'duration', 'isi'])
if zf['heliox'].isna().any():
    raise ValueError(f'{int(zf["heliox"].isna().sum())} zebra finch syllables have no heliox flag')
# heliox holds True/False for some birds and 1.0/0.0 for others; both mean the same thing
heliox = zf['heliox'].astype(float).astype(bool).to_numpy()
if 'condition' in zf.columns:
    disagree = int((zf['condition'].astype(str).str.lower().str.contains('heliox').to_numpy() != heliox).sum())
    if disagree:
        print(f'[CHECK] {disagree} syllables where the heliox flag and the condition column disagree; '
              'the flag is used, as in the original analysis')
raw_names = zf['indv'].astype(str)
full_names = raw_names.replace(ZF_FULL_NAME)
merged = full_names.groupby(full_names).apply(lambda s: raw_names[s.index].nunique())
if (merged > 1).any():
    raise ValueError(f'two names in the table become one bird: {list(merged[merged > 1].index)}')
zf_syl = pd.DataFrame({
    'bird': full_names.to_numpy(),
    'song_id': zf['song_id'].astype(str).to_numpy(),
    'condition': np.where(heliox, 'heliox', 'regair'),
    'label': zf['label'].to_numpy(),
    'intro': (zf['label'] == INTRO_LABEL).to_numpy(),
    'duration': pd.to_numeric(zf['duration'], errors='coerce').to_numpy(),
    'isi': pd.to_numeric(zf['isi'], errors='coerce').to_numpy()})
del zf
gc.collect()

song_rows, zf_motif_orders = [], {}
for bird, b in zf_syl.groupby('bird', sort=False):
    order = motif_order(b['label'].tolist())
    zf_motif_orders[bird] = [py(x) for x in order]
    for song_id, s in b.groupby('song_id', sort=False):
        song_rows.append(dict(bird=bird, song_id=song_id, condition=s['condition'].iloc[0],
                              mixed=s['condition'].nunique() > 1, n_syllables=len(s),
                              motifs=motifs_in_song(s['label'].tolist(), order[0], len(order))))
zf_songs = pd.DataFrame(song_rows)
if zf_songs['mixed'].any():
    print(f'[CHECK] {int(zf_songs["mixed"].sum())} songs hold syllables of both conditions; '
          'each takes the condition of its first syllable, as in the original analysis')
zf_songs = zf_songs.drop(columns='mixed')

print(f'zebra finch: {len(zf_syl)} syllables, {int(zf_syl["intro"].sum())} of them intro notes; {len(zf_songs)} songs')
print('motif of each bird (syllable labels in order):', zf_motif_orders)
print(zf_songs.groupby(['bird', 'condition'])['motifs'].agg(['size', 'mean']).unstack().round(2).to_string())

# ---- starlings: the six syllable tables the starling LME notebook read (Cell F1) ---------------
frames = []
for bird, rel in STARLING_FEATURE_FILES.items():
    df = read_columns(os.path.join(BASE_DIR, rel),
                      ['condition', 'duration', 'time_between_syllables', 'bout_index', 'syllables_sequence_pos'])
    frames.append(pd.DataFrame({
        'bird': bird,
        'condition': df['condition'].astype(str).str.lower().to_numpy(),
        # bout_index holds each song's wav path here; as text, Path objects and strings group alike
        'bout_key': df['bout_index'].map(lambda v: None if pd.isna(v) else str(v)).to_numpy(),
        'position': pd.to_numeric(df['syllables_sequence_pos'], errors='coerce').to_numpy(),
        'duration': pd.to_numeric(df['duration'], errors='coerce').to_numpy(),
        'isi': pd.to_numeric(df['time_between_syllables'], errors='coerce').to_numpy()}))
st = pd.concat(frames, ignore_index=True)
del frames, df
left_out = st.loc[~st['condition'].isin(COND_ORDER), 'condition'].value_counts().to_dict()
st = st[st['condition'].isin(COND_ORDER)].reset_index(drop=True)     # drops 'regairb', as originally

# A bout is one wav path, across all birds, as in the original; say so if two birds share one.
shared = int((st.dropna(subset=['bout_key']).groupby('bout_key')['bird'].nunique() > 1).sum())
if shared:
    print(f'[CHECK] {shared} bout ids are used by more than one bird')
st['bout'] = pd.factorize(st['bout_key'])[0]                          # -1 where there is no bout id
last = st['bout'].map(st[st['bout'] >= 0].groupby('bout')['position'].max())
# 0 at the first syllable of the bout, 1 at the last; single-syllable bouts get none
st['position_norm'] = np.where(last > 0, st['position'] / last, np.nan)

print(f'\nstarling: {len(st)} syllables in {st.loc[st["bout"] >= 0, "bout"].nunique()} bouts'
      + (f'   [left out: {left_out}]' if left_out else ''))
print(st.groupby(['bird', 'condition']).size().unstack(fill_value=0).to_string())

# ---- check and save -----------------------------------------------------------------------------
song_structure = {
    'zf_syllables': zf_syl[['bird', 'song_id', 'condition', 'label', 'intro', 'duration', 'isi']].astype(
        {'song_id': 'category'}),
    'zf_songs': zf_songs,
    'starling_syllables': st[['bird', 'condition', 'bout', 'position', 'position_norm', 'duration', 'isi']],
}
for name, df in song_structure.items():
    require_complete(df, name, 'bird', 'condition', species='starling' if name.startswith('starling') else 'zebra_finch')
    assert_portable(name, df)

song_structure['meta'] = {
    'created': datetime.datetime.now().isoformat(timespec='seconds'),
    'sources': {'zebra finch': os.path.basename(ZF_SYL_PATH),
                'starling': {b: os.path.basename(r) for b, r in STARLING_FEATURE_FILES.items()}},
    'zf_motif_orders': zf_motif_orders,
    'notes': {'intro': f'label == {INTRO_LABEL}',
              'motifs': 'spacing count: appearances of the motif\'s first syllable at least one motif length '
                        'after the last counted one (the first appearance is not counted)',
              'duration, isi': 'seconds, both species',
              'bout': 'starling bout id (one wav file); -1 where the table gives none',
              'position_norm': 'starling: position / last position in the bout; NaN for single-syllable bouts',
              'conditions left out': left_out},
    'versions': {'numpy': np.__version__, 'pandas': pd.__version__},
}
with open(SONG_STRUCTURE_DATA_PATH, 'wb') as f:
    pickle.dump(song_structure, f, protocol=4)
print(f'\nsaved {SONG_STRUCTURE_DATA_PATH}  ({os.path.getsize(SONG_STRUCTURE_DATA_PATH) / 1e6:.1f} MB)')

zebra finch: 63728 syllables, 8658 of them intro notes; 2027 songs
motif of each bird (syllable labels in order): {'z_p7y2_23': [2, 3, 0, 1], 'z_y11c13_24': [3, 4, 1, 2], 'z_r11g11_24': [2, 3, 4, 5, 6, 0, 1], 'z_c8c19_24': [0, 1, 2, 3, 4], 'z_p4g13_23': [2, 3], 'z_g14g16_24': [3, 4, 5]}
              size          mean       
condition   heliox regair heliox regair
bird                                   
z_c8c19_24     137    103   4.71   5.32
z_g14g16_24    284    248   8.47  11.53
z_p4g13_23     258    263   4.75   7.29
z_p7y2_23       38    123   0.68   2.02
z_r11g11_24     81    143   1.85   2.07
z_y11c13_24    134    215   9.89  10.61

starling: 95063 syllables in 2679 bouts   [left out: {'regairb': 3537}]
condition    heliox  regair
bird                       
s_b1349h_23    8232    4451
s_b1354_25     2238   15100
s_b1364h_25    6827   42928
s_b1481h_24     696    1640
s_b1484h_24    2502    4426
s_b1556h_24    1233    4790
zf_syllables: complete (63728 rows)
zf_songs: complete 

In [50]:
# Cell K1: k-gram analysis (Fig. 4 H-I, Supp. Fig. S5), settings and the starling syllable sequences
# Feeds kgram_figures.ipynb. Cells K1-K5 redo what the k-gram notebook computed, from the same
# tables, and K5 compares the k-grams selected here with the ones that notebook saved.
# Run after Cells 1-3 (LABCOL, BOUTCOL, POSCOL, STARLING_SYL_DIR), F1 (read_columns) and 4.
from collections import Counter, defaultdict

KGRAM_DATA_PATH = _path('HELIOX_KGRAM_DATA', os.path.join(DERIVED, 'kgram_data.pkl'))

# The k-gram notebook read the augmented tables under figures/starling_runs, with bird 1364 under
# s_b1364h_25: the table its clusters were made on.
KGRAM_BIRDS = ['s_b1349h_23', 's_b1354_25', 's_b1364h_25', 's_b1481h_24', 's_b1484h_24', 's_b1556h_24']
KS = [3, 4, 5, 6, 7, 8]
PREVALENCE = 0.01      # a selected k-gram occurs in at least 1% of the condition's bouts ...
FOLD_CUT = 2.0         # ... and at least twice as often as a first-order Markov chain predicts
MIN_SELECTED = 10      # a bird enters the ISI model at a given k with this many selected k-grams in each condition
N_SPLITS = 50          # split-half resamples per bird, condition and k (Supp. S5B)
SOFTMATCH_MAX = 600    # a larger selected set is subsampled before soft matching, as in the original
SPLIT_SEED = 0
THRESHOLD_SWEEP = [0.001, 0.005, 0.01, 0.02, 0.03]     # Supp. S5F
ISI_SOURCE = 'time_between_syllables'
# The k-gram notebook counted 'regairb' (regular air recorded after heliox) as regular air,
# unlike the other analyses, which leave it out. True reproduces Table 2 and Supp. S5.
REGAIRB_AS_REGAIR = True


def bout_rank(values):
    """Each bout's place in sorted order. The original sorted the raw bout_index values (wav paths,
    as Path objects or text), and that order decides the split-half draws, so the raw values are
    sorted here too rather than their text."""
    uniq = sorted(set(v for v in values if not pd.isna(v)))
    rank = {v: i for i, v in enumerate(uniq)}
    return np.array([rank.get(v, -1) if not pd.isna(v) else -1 for v in values])


frames = []
for bird in KGRAM_BIRDS:
    path = os.path.join(STARLING_SYL_DIR, bird, f'{bird}_syllable_df_augmented.pkl')
    df = read_columns(path, ['condition', BOUTCOL, POSCOL, LABCOL, ISI_SOURCE])
    missing = {'condition', BOUTCOL, POSCOL, LABCOL, ISI_SOURCE} - set(df.columns)
    if missing:
        raise KeyError(f'{bird}: no {sorted(missing)} in {os.path.basename(path)}')
    cond = df['condition'].astype(str).str.lower()
    frames.append(pd.DataFrame({
        'bird': bird,
        'raw_condition': cond.to_numpy(),
        'condition': np.where(cond.str.contains('heliox'), 'heliox', 'regair'),   # the original's rule
        'bout': bout_rank(df[BOUTCOL].tolist()),
        'position': pd.to_numeric(df[POSCOL], errors='coerce').to_numpy(),
        'label': pd.to_numeric(df[LABCOL], errors='coerce').to_numpy(),
        'isi': pd.to_numeric(df[ISI_SOURCE], errors='coerce').to_numpy()}))
kg_syl = pd.concat(frames, ignore_index=True)
del frames, df
gc.collect()

print('syllables per bird and condition, as written in the tables:')
print(pd.crosstab(kg_syl['bird'], kg_syl['raw_condition']).to_string())
if not REGAIRB_AS_REGAIR:
    kg_syl = kg_syl[kg_syl['raw_condition'].isin(COND_ORDER)]
n_regairb = int((kg_syl['raw_condition'] == 'regairb').sum())
# no bout, position or label: left out; unclustered syllables (label -1): left out, as originally
kg_syl = kg_syl[(kg_syl['bout'] >= 0) & kg_syl['position'].notna() & kg_syl['label'].notna()]
kg_syl = kg_syl[kg_syl['label'] != -1]
kg_syl['label'] = kg_syl['label'].astype(int)
kg_syl = kg_syl.sort_values(['bird', 'condition', 'bout', 'position']).reset_index(drop=True)

# One label sequence per bout, for each bird x condition, in the original's order
# (birds alphabetically, heliox before regair, bouts in sorted order).
bout_seqs = {}
for key, g in kg_syl.groupby(['bird', 'condition'], sort=False):
    bout_seqs[key] = {b: s['label'].tolist() for b, s in g.groupby('bout', sort=False)}

print(f'\n{len(kg_syl)} clustered syllables in {sum(len(v) for v in bout_seqs.values())} bouts'
      + (f'; {n_regairb} of them regairb, counted as regular air' if n_regairb else ''))
print(kg_syl.groupby(['bird', 'condition'])['bout'].nunique().unstack().rename_axis(None, axis=1)
      .add_suffix(' bouts').to_string())

syllables per bird and condition, as written in the tables:
raw_condition  heliox  regair
bird                         
s_b1349h_23      8232    4451
s_b1354_25       2238   17813
s_b1364h_25      6827   43752
s_b1481h_24       696    1640
s_b1484h_24      2502    4426
s_b1556h_24      1233    4790

60206 clustered syllables in 2761 bouts
             heliox bouts  regair bouts
bird                                   
s_b1349h_23           389           218
s_b1354_25             81           622
s_b1364h_25           168           838
s_b1481h_24            54            76
s_b1484h_24            80           119
s_b1556h_24            25            91


In [51]:
# Cell K2: every k-gram each bird sang, its bout prevalence and its fold enrichment over a
# first-order Markov chain; which pass both criteria; and the size of each tier of Fig. 4I
def markov_model(bouts):
    """Syllable probabilities and first-order transition probabilities of one bird x condition."""
    counts, transitions, n = Counter(), defaultdict(Counter), 0
    for seq in bouts.values():
        counts.update(seq)
        n += len(seq)
        for a, b in zip(seq[:-1], seq[1:]):
            transitions[a][b] += 1
    marginal = {s: c / n for s, c in counts.items()}
    p_next = {}
    for a, nxt in transitions.items():
        total = sum(nxt.values())
        for b, c in nxt.items():
            p_next[(a, b)] = c / total
    return marginal, p_next


def markov_prob(kg, marginal, p_next):
    """Probability of a k-gram under the Markov chain: P(s1) P(s2|s1) ... P(sk|sk-1)."""
    p = marginal.get(kg[0], 0.0)
    for a, b in zip(kg[:-1], kg[1:]):
        p *= p_next.get((a, b), 0.0)
        if p == 0.0:
            return 0.0
    return p


def markov_reachable(marginal, p_next, k):
    """How many k-grams can be built by chaining only transitions the bird used at least once."""
    nxt = defaultdict(list)
    for a, b in p_next:
        nxt[a].append(b)
    ways = {s: 1 for s in marginal}                  # sequences of length 1 starting at each syllable
    for _ in range(k - 1):
        ways = {s: sum(ways.get(b, 0) for b in nxt.get(s, [])) for s in marginal}
    return sum(ways.values())


rows, tiers = [], []
for (bird, cond), bouts in bout_seqs.items():
    marginal, p_next = markov_model(bouts)
    for k in KS:
        observed, in_bouts, n_slots = Counter(), Counter(), 0
        for seq in bouts.values():
            grams = [tuple(seq[i:i + k]) for i in range(len(seq) - k + 1)]
            n_slots += len(grams)
            observed.update(grams)
            in_bouts.update(set(grams))                  # prevalence counts each bout once
        for kg, n_obs in observed.items():
            rows.append((bird, cond, k, kg, in_bouts[kg], len(bouts), n_obs,
                         markov_prob(kg, marginal, p_next) * n_slots))
        tiers.append(dict(bird=bird, condition=cond, k=k, n_types=len(marginal),
                          theoretical_max=len(marginal) ** k,
                          markov_reachable=markov_reachable(marginal, p_next, k)))

kgrams = pd.DataFrame(rows, columns=['bird', 'condition', 'k', 'kg', 'n_bouts_with', 'n_bouts', 'observed', 'expected'])
kgrams['pct_bouts'] = kgrams['n_bouts_with'] / kgrams['n_bouts']
# a k-gram needing a transition the bird never made has expected count 0: infinitely enriched
kgrams['fold'] = np.where(kgrams['expected'] > 1e-12, kgrams['observed'] / kgrams['expected'].clip(lower=1e-300), np.inf)
kgrams['selected'] = (kgrams['pct_bouts'] >= PREVALENCE) & (kgrams['fold'] >= FOLD_CUT)
kgrams['kgram'] = kgrams['kg'].map(lambda kg: '-'.join(str(x) for x in kg))    # as text in the pickle

selected = {key: set(g['kg']) for key, g in kgrams[kgrams['selected']].groupby(['bird', 'condition', 'k'])}
del rows

counts = kgrams.groupby(['bird', 'condition', 'k']).agg(n_identified=('kg', 'size'), n_selected=('selected', 'sum'))
kg_space = pd.DataFrame(tiers).merge(counts.reset_index(), on=['bird', 'condition', 'k'], how='left')
bad = kg_space[(kg_space['n_selected'] > kg_space['n_identified'])
               | (kg_space['n_identified'] > kg_space['markov_reachable'])
               | (kg_space['markov_reachable'] > kg_space['theoretical_max'])]
if len(bad):
    raise RuntimeError(f'{len(bad)} bird x condition x k rows break the order theoretical max >= '
                       'Markov-reachable >= identified >= selected')

print(f'{len(kgrams)} bird x condition x k-gram rows; {int(kgrams["selected"].sum())} selected')
print('\nmean per bird (heliox | regair):')
print(kg_space.groupby(['k', 'condition'])[['markov_reachable', 'n_identified', 'n_selected']].mean()
      .unstack().round(1).to_string())

174540 bird x condition x k-gram rows; 41920 selected

mean per bird (heliox | regair):
          markov_reachable               n_identified         n_selected       
condition           heliox        regair       heliox  regair     heliox regair
k                                                                              
3             5.072500e+03  6.260300e+03        920.3  1357.5      238.8  251.5
4             7.457680e+04  8.170520e+04       1327.8  2463.0      473.3  560.2
5             1.239528e+06  1.106248e+06       1528.3  3451.3      594.3  707.5
6             2.208661e+07  1.527033e+07       1585.2  4126.8      645.8  745.8
7             4.100585e+08  2.147333e+08       1566.0  4526.7      657.5  743.5
8             7.783870e+09  3.063504e+09       1509.3  4727.7      649.0  719.3


In [52]:
# Cell K3: ISI inside and outside the selected k-grams (Fig. 4H, Table 2, Supp. S5A)
# One row per bird x condition x k: the median ISI of transitions inside an occurrence of a
# selected k-gram and of all other transitions. The figure notebook fits the models.
def inside_mask(labels, k, chosen):
    """True for each syllable whose ISI (the gap to the next syllable) falls inside an occurrence
    of a selected k-gram, i.e. the first k-1 syllables of every occurrence."""
    inside = np.zeros(len(labels), dtype=bool)
    for i in range(len(labels) - k + 1):
        if tuple(labels[i:i + k]) in chosen:
            inside[i:i + k - 1] = True
    return inside


# As in the original, syllables without an ISI (mostly the last one of a bout) are dropped before
# k-grams are matched here, so an occurrence that ends on a bout's last syllable is not marked.
with_isi = kg_syl[kg_syl['isi'].notna()]
rows = []
for (bird, cond), g in with_isi.groupby(['bird', 'condition'], sort=False):
    labels_by_bout = [s['label'].tolist() for _, s in g.groupby('bout', sort=False)]   # rows are in bout order
    isi = g['isi'].to_numpy()
    for k in KS:
        chosen = selected.get((bird, cond, k), set())
        inside = np.concatenate([inside_mask(labels, k, chosen) for labels in labels_by_bout])
        rows.append(dict(bird=bird, condition=cond, k=k, n_selected=len(chosen),
                         n_inside=int(inside.sum()), n_outside=int((~inside).sum()),
                         median_isi_inside=np.median(isi[inside]) if inside.any() else np.nan,
                         median_isi_outside=np.median(isi[~inside]) if (~inside).any() else np.nan))
isi_by_k = pd.DataFrame(rows).sort_values(['k', 'bird', 'condition']).reset_index(drop=True)

# a bird enters the model at a k only with enough selected k-grams in BOTH conditions
fewest = isi_by_k.groupby(['bird', 'k'])['n_selected'].transform('min')
isi_by_k['in_model'] = fewest >= MIN_SELECTED

print('birds in the model at each k:', isi_by_k[isi_by_k['in_model']].groupby('k')['bird'].nunique().to_dict())
print(isi_by_k.pivot_table(index=['k', 'bird'], columns='condition',
                           values=['median_isi_inside', 'median_isi_outside']).round(4).head(12).to_string())

birds in the model at each k: {3: 6, 4: 6, 5: 6, 6: 6, 7: 5, 8: 5}
              median_isi_inside        median_isi_outside       
condition                heliox regair             heliox regair
k bird                                                          
3 s_b1349h_23            0.2295  0.157             0.1760  0.107
  s_b1354_25             0.1260  0.111             0.0820  0.091
  s_b1364h_25            0.2270  0.072             0.1970  0.084
  s_b1481h_24            0.1290  0.086             0.1810  0.063
  s_b1484h_24            0.1415  0.095             0.1500  0.150
  s_b1556h_24            0.0690  0.087             0.0610  0.068
4 s_b1349h_23            0.1730  0.151             0.1860  0.107
  s_b1354_25             0.0980  0.134             0.0800  0.080
  s_b1364h_25            0.1860  0.068             0.2335  0.095
  s_b1481h_24            0.1420  0.073             0.2210  0.068
  s_b1484h_24            0.1440  0.129             0.1900  0.153
  s_b1556h_24          

In [53]:
# Cell K4: split-half stability of the selected k-grams (Supp. S5B)
# Slow: a few minutes. Each bird x condition x k: the bouts are split in random halves N_SPLITS times,
# k-grams are selected in each half, and the two halves' sets are compared. As in the original, the
# prevalence criterion is reapplied in each half while fold enrichment comes from all of the bouts.
# The draws follow the original order exactly (one generator, seed 0, same loop order), so the
# numbers match the paper rather than being one more random draw.
import warnings


def kgram_bout_matrix(bouts, k):
    """The k-grams seen in these bouts, and a bouts x k-grams table of which bout holds which."""
    all_kgrams, bout_sets = set(), []
    for seq in bouts.values():
        s = {tuple(seq[i:i + k]) for i in range(len(seq) - k + 1)} if len(seq) >= k else set()
        bout_sets.append(s)
        all_kgrams.update(s)
    kgram_list = list(all_kgrams)
    index = {kg: i for i, kg in enumerate(kgram_list)}
    M = np.zeros((len(bout_sets), len(kgram_list)), dtype=bool)
    for i, s in enumerate(bout_sets):
        for kg in s:
            M[i, index[kg]] = True
    return kgram_list, M


def selected_in(rows, kgram_list, M, fold_ok):
    """The k-grams selected among the bouts in `rows` (built in list order, as the original did)."""
    if len(rows) == 0:
        return set()
    n_with = M[rows].sum(axis=0)
    keep = (n_with > 0) & (n_with / len(rows) >= PREVALENCE) & fold_ok
    return {kgram_list[i] for i in np.flatnonzero(keep)}


def best_match(set_a, set_b, k, rng):
    """Soft stability: for each k-gram of one half, the share of positions it has in common with
    its closest k-gram in the other half, averaged."""
    if not set_a or not set_b:
        return np.nan
    a, b = list(set_a), list(set_b)
    if len(a) > SOFTMATCH_MAX:
        a = [a[i] for i in rng.choice(len(a), size=SOFTMATCH_MAX, replace=False)]
    if len(b) > SOFTMATCH_MAX:
        b = [b[i] for i in rng.choice(len(b), size=SOFTMATCH_MAX, replace=False)]
    A, B = np.asarray(a, dtype=np.int64), np.asarray(b, dtype=np.int64)
    return float(((A[:, None, :] == B[None, :, :]).sum(axis=2) / k).max(axis=1).mean())


fold_of = {key: dict(zip(g['kg'], g['fold'])) for key, g in kgrams.groupby(['bird', 'condition'], sort=False)}
rng = np.random.default_rng(SPLIT_SEED)
rows = []
for (bird, cond), bouts in bout_seqs.items():
    fold = fold_of.get((bird, cond), {})
    for k in KS:
        kgram_list, M = kgram_bout_matrix(bouts, k)
        n = M.shape[0]
        if n < 2 or not kgram_list:
            continue
        fold_ok = np.array([fold.get(kg, 0.0) >= FOLD_CUT for kg in kgram_list])
        exact, soft = [], []
        for _ in range(N_SPLITS):
            perm = rng.permutation(n)
            half = n // 2
            a = selected_in(perm[:half], kgram_list, M, fold_ok)
            b = selected_in(perm[half:2 * half], kgram_list, M, fold_ok)
            union = a | b
            exact.append(len(a & b) / len(union) if union else np.nan)
            soft.append(best_match(a, b, k, rng))
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)       # all-NaN means are left as NaN
            rows.append(dict(bird=bird, condition=cond, k=k,
                             n_selected=len(selected_in(np.arange(n), kgram_list, M, fold_ok)),
                             exact_match=np.nanmean(exact), best_match=np.nanmean(soft)))
    print(f'  {bird} {cond}: done')
stability = pd.DataFrame(rows)
print(stability.groupby(['condition', 'k'])[['exact_match', 'best_match']].mean().unstack(0).round(3).to_string())

  s_b1349h_23 heliox: done
  s_b1349h_23 regair: done
  s_b1354_25 heliox: done
  s_b1354_25 regair: done
  s_b1364h_25 heliox: done
  s_b1364h_25 regair: done
  s_b1481h_24 heliox: done
  s_b1481h_24 regair: done
  s_b1484h_24 heliox: done
  s_b1484h_24 regair: done
  s_b1556h_24 heliox: done
  s_b1556h_24 regair: done
          exact_match        best_match       
condition      heliox regair     heliox regair
k                                             
3               0.167  0.308      0.677  0.751
4               0.119  0.254      0.702  0.746
5               0.071  0.207      0.673  0.727
6               0.042  0.178      0.639  0.708
7               0.022  0.152      0.587  0.704
8               0.013  0.111      0.559  0.700


In [54]:
# Cell K5: check and save the k-gram data
# Same k-grams as the original notebook selected? It saved them to MOTIF_PKL (Cell 1).
if os.path.exists(MOTIF_PKL):
    saved = load_pickle(MOTIF_PKL)['final_motif_sets']
    compared = []
    for key in sorted(set(saved) | set(selected)):
        old = {tuple(int(x) for x in kg) for kg in saved.get(key, set())}
        new = selected.get(key, set())
        compared.append(dict(bird=key[0], condition=key[1], k=key[2], saved=len(old), here=len(new),
                             in_both=len(old & new)))
    compared = pd.DataFrame(compared)
    differ = compared[(compared['saved'] != compared['in_both']) | (compared['here'] != compared['in_both'])]
    if differ.empty:
        print(f'selected k-grams identical to {os.path.basename(MOTIF_PKL)} '
              f'({len(compared)} bird x condition x k sets)')
    else:
        print(f'[CHECK] {len(differ)} of {len(compared)} selected sets differ from {os.path.basename(MOTIF_PKL)}:')
        print(differ.to_string(index=False))
    del saved
else:
    print(f'no {MOTIF_PKL} to compare against')

kgram_data = {
    'kgrams': kgrams[['bird', 'condition', 'k', 'kgram', 'n_bouts_with', 'n_bouts', 'pct_bouts',
                      'observed', 'expected', 'fold', 'selected']].astype({'bird': 'category', 'condition': 'category'}),
    'space': kg_space,
    'isi_by_k': isi_by_k,
    'stability': stability,
}
for name, df in kgram_data.items():
    require_complete(df.astype({'bird': str, 'condition': str}), name, 'bird', 'condition', species='starling')
    assert_portable(name, df)

kgram_data['meta'] = {
    'created': datetime.datetime.now().isoformat(timespec='seconds'),
    'source': f'<bird>/<bird>_syllable_df_augmented.pkl under figures/starling_runs, labels {LABCOL}',
    'settings': dict(KS=KS, PREVALENCE=PREVALENCE, FOLD_CUT=FOLD_CUT, MIN_SELECTED=MIN_SELECTED, N_SPLITS=N_SPLITS,
                     SOFTMATCH_MAX=SOFTMATCH_MAX, SPLIT_SEED=SPLIT_SEED, THRESHOLD_SWEEP=THRESHOLD_SWEEP,
                     REGAIRB_AS_REGAIR=REGAIRB_AS_REGAIR, ISI_SOURCE=ISI_SOURCE),
    'notes': {'kgrams': 'every k-gram each bird sang in each condition, unclustered syllables left out; kgram '
                        'is the label sequence joined by "-"; pct_bouts = n_bouts_with / n_bouts; expected = '
                        'first-order Markov count; fold = observed / expected (inf when expected is 0)',
              'space': 'Fig. 4I tiers per bird x condition x k',
              'isi_by_k': 'median ISI (s) inside / outside selected k-grams; in_model = at least MIN_SELECTED '
                          'selected k-grams in both conditions',
              'stability': 'split-half exact (Jaccard) and best-match stability, mean over N_SPLITS splits',
              'regairb': 'counted as regular air' if REGAIRB_AS_REGAIR else 'left out'},
    'versions': {'numpy': np.__version__, 'pandas': pd.__version__},
}
with open(KGRAM_DATA_PATH, 'wb') as f:
    pickle.dump(kgram_data, f, protocol=4)
print(f'\nsaved {KGRAM_DATA_PATH}  ({os.path.getsize(KGRAM_DATA_PATH) / 1e6:.1f} MB)')

selected k-grams identical to starling_final_motif_sets.pkl (72 bird x condition x k sets)
kgrams: complete (174540 rows)
space: complete (72 rows)
isi_by_k: complete (72 rows)
stability: complete (72 rows)

saved /mnt/cube/lstanwic/umap_avgn_shared/lauren_umap/heliox_processing/data/derived/kgram_data.pkl  (13.6 MB)


## Fig. 5: within-song effects, latent space and exposure course

L1 runs no analysis: it collects the saved results of the VAE analysis into one file.

In [55]:
# Cell W1: within-song data, settings
# Feeds within_song_figures.ipynb: Fig. 5 A-E and Supp. Fig. S6 (starlings only).
import gc
from scipy.signal import get_window

WITHIN_SONG_DATA_PATH = _path('HELIOX_WITHIN_SONG_DATA', os.path.join(DERIVED, 'within_song_data.pkl'))
# one table per starling: that bird's syllable table with the UMAP clustering added,
# <bird>/<bird>_syllable_df_augmented.pkl
AUGMENTED_DIR = _path('HELIOX_STARLING_SYL_DIR', os.path.join(PROC_DIR, 'figures'))

MIN_SYLLABLES = 10       # a song enters the tempo analysis with at least this many syllables
MIN_SONG_LEN_S = 10.0    # ... and at least this long from first onset to last offset
TEMPO_SMOOTH_N = 5       # Hann window, in intervals, applied to each song's tempo trace
JUMP_THRESH_MS = 15.0    # a step between successive tempo values larger than this is a jump
PAUSE_FACTOR = 2.0       # an interval longer than this many song medians is a pause

# The two songs drawn in Fig. 5D, by wav file name.
TEMPO_EXAMPLES = {'low': 's_b1556h_24_2024-03-27_15-00-01-01_1297865_1347030_song_regair.wav',
                  'high': 's_b1556h_24_2024-03-28_12-00-01-01_1752940_1795615_song_heliox.wav'}

print('within-song data ->', WITHIN_SONG_DATA_PATH)

within-song data -> /mnt/cube/lstanwic/umap_avgn_shared/lauren_umap/heliox_processing/data/derived/within_song_data.pkl


In [56]:
# Cell W2: the six starling syllable tables, cut down to the columns these figures use
# Each table carries every spectrogram, so one bird is read at a time and only the eight
# columns needed here are kept.
_POSIX = re.compile(r"PosixPath\(['\"](.+?)['\"]\)")


def path_text(p):
    """A wav path as plain text, whether it was stored as text, as a Path, or as 'PosixPath(...)'."""
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return None
    m = _POSIX.search(str(p))
    return m.group(1) if m else str(p)


WANTED = ['file', 'condition', 'onsets_ms', 'offsets_ms', BOUTCOL, POSCOL, 't_pos', LABCOL]
frames = []
for bird in STARLING_BIRDS:
    pkl = os.path.join(AUGMENTED_DIR, bird, f'{bird}_syllable_df_augmented.pkl')
    raw = pd.read_pickle(pkl)
    # by position, first occurrence of each name: one table carries a column name twice
    first = [i for i, c in enumerate(raw.columns) if c in WANTED and c not in list(raw.columns[:i])]
    df = raw.iloc[:, first].copy()
    del raw
    gc.collect()
    files = df['file'].map(path_text)
    one = pd.DataFrame({
        'bird': bird,
        'condition': df['condition'].astype(str).str.lower(),
        # same path clean-up as the tempo analysis, so that songs sort in the same order
        'file': files.str.replace('jeff_umap', 'lauren_umap', regex=False)
                     .str.replace('starling_rhythm_mfdfa', 'heliox_processing', regex=False),
        'bout': df[BOUTCOL].map(path_text) if BOUTCOL in df.columns else files,
        'position': df['t_pos'] if 't_pos' in df.columns else df[POSCOL],
        'onset_s': pd.to_numeric(df['onsets_ms'], errors='coerce'),    # misnamed: these columns
        'offset_s': pd.to_numeric(df['offsets_ms'], errors='coerce'),  # hold seconds in the starling tables
        'label': df[LABCOL],
    })
    # the other analyses read a copy of this table under figures/starling_runs: say so if it is not the same file
    twin = os.path.join(STARLING_SYL_DIR, bird, os.path.basename(pkl))
    differs = os.path.exists(twin) and os.path.getsize(twin) != os.path.getsize(pkl)
    print(f'  {bird}: {len(one):>6d} syllables, {one["file"].nunique():>4d} wav files, table dated '
          f'{datetime.datetime.fromtimestamp(os.path.getmtime(pkl)):%Y-%m-%d}'
          + ('   [CHECK] the copy under figures/starling_runs is a different size' if differs else ''))
    frames.append(one)
ws_syl = pd.concat(frames, ignore_index=True)
ws_syl['song_file'] = ws_syl['file'].map(lambda p: os.path.basename(p) if p else None)
del frames, df, one
gc.collect()

other = sorted(set(ws_syl['condition']) - {'regair', 'heliox'})
if other:
    raise ValueError(f'conditions other than regair and heliox in the syllable tables: {other}')
if ws_syl['offset_s'].median() > 600:
    raise ValueError('onsets_ms / offsets_ms look like milliseconds here; the tempo metrics expect seconds')
if ws_syl.groupby('bird')['song_file'].nunique().sum() != ws_syl.groupby('bird')['file'].nunique().sum():
    raise ValueError('two wav files of one bird share a file name, so songs cannot be named by file name')
print(f'{len(ws_syl)} syllables in all; {int((ws_syl["label"] < 0).sum())} of them unclustered (label -1)')

  s_b1556h_24:   6023 syllables,  117 wav files, table dated 2026-08-26
  s_b1484h_24:   6928 syllables,  199 wav files, table dated 2026-08-26
  s_b1364_25:  14069 syllables,  294 wav files, table dated 2026-08-26
  s_b1354_25:  20051 syllables,  703 wav files, table dated 2026-08-26
  s_b1481h_24:   2336 syllables,  131 wav files, table dated 2026-08-26
  s_b1349h_23:  12683 syllables,  608 wav files, table dated 2026-08-26
62090 syllables in all; 11388 of them unclustered (label -1)


In [57]:
# Cell W3: tempo metrics, one row per song (Fig. 5 D-E)
# Tempo is the interval from each syllable onset to the next, in ms, lightly smoothed.
def tempo_trace(onsets_s, step=1, smooth_n=TEMPO_SMOOTH_N):
    """Onset times and the smoothed onset-to-onset intervals (ms) that start at them."""
    t = np.asarray(onsets_s, float)
    t = np.sort(t[np.isfinite(t)])
    if len(t) < step + 1:
        return np.array([]), np.array([])
    dt = t[step:] - t[:-step]
    if smooth_n > 1 and len(dt) >= 3:
        w = get_window('hann', min(int(smooth_n), len(dt)))
        dt = np.convolve(dt, w / (np.sum(w) + 1e-12), mode='same')
    return t[:-step], dt * 1000.0


def tempo_metrics(tempo_ms):
    y = tempo_ms[np.isfinite(tempo_ms)]
    if len(y) < 3:
        return dict(tempo_roughness_ms=np.nan, tempo_jump_rate=np.nan, tempo_pause_frac=np.nan)
    step = np.abs(np.diff(y))
    return dict(tempo_roughness_ms=float(np.nanmedian(step)),                  # typical step size
                tempo_jump_rate=float(np.mean(step > JUMP_THRESH_MS)),          # share of steps that are jumps
                tempo_pause_frac=float(np.mean(y > PAUSE_FACTOR * float(np.nanmedian(y)))))


rows = []
for bird in STARLING_BIRDS:                                   # bird order of the original analysis
    for path, g in ws_syl[ws_syl['bird'] == bird].groupby('file'):
        song_len = g['offset_s'].max() - g['onset_s'].min()
        if len(g) < MIN_SYLLABLES or not (song_len >= MIN_SONG_LEN_S):
            continue
        onsets = g['onset_s'].to_numpy(float)
        onsets = onsets[np.isfinite(onsets)]
        row = dict(bird=bird, condition=g['condition'].mode().iat[0], file=path, song_file=os.path.basename(path),
                   song_len_s=song_len, n_syllables=len(g))
        row.update(tempo_metrics(tempo_trace(onsets)[1]) if len(onsets) >= 3 else tempo_metrics(np.array([])))
        rows.append(row)
ws_tempo = pd.DataFrame(rows)
print('songs per bird x condition:')
print(ws_tempo.groupby(['bird', 'condition']).size().unstack(fill_value=0).to_string())

# Same table as the one the original tempo analysis saved? Checked song by song when that file exists.
TEMPO_COLS = ['tempo_roughness_ms', 'tempo_jump_rate', 'tempo_pause_frac']
saved_path = os.path.join(DERIVED, 'tempo_syl.pickle')
if os.path.exists(saved_path):
    saved = pd.read_pickle(saved_path)
    saved_names = saved['wav_path'].map(lambda p: os.path.basename(path_text(p)))
    if len(saved) != len(ws_tempo) or set(saved_names) != set(ws_tempo['song_file']):
        raise RuntimeError(f'tempo table has {len(ws_tempo)} songs, tempo_syl.pickle has {len(saved)}; '
                           f'{len(set(ws_tempo["song_file"]) - set(saved_names))} only here, '
                           f'{len(set(saved_names) - set(ws_tempo["song_file"]))} only there')
    old = saved.assign(song_file=saved_names).set_index('song_file').loc[ws_tempo['song_file']]
    gap = np.abs(old[TEMPO_COLS].to_numpy(float) - ws_tempo[TEMPO_COLS].to_numpy(float))
    worst = np.nanmax(gap) if np.isfinite(gap).any() else 0.0
    nan_same = np.array_equal(np.isnan(old[TEMPO_COLS].to_numpy(float)), np.isnan(ws_tempo[TEMPO_COLS].to_numpy(float)))
    if worst > 1e-9 or not nan_same or (old['condition'].to_numpy() != ws_tempo['condition'].to_numpy()).any():
        raise RuntimeError(f'tempo metrics differ from tempo_syl.pickle (largest difference {worst:.3g})')
    same_order = (saved_names.to_numpy() == ws_tempo['song_file'].to_numpy()).all()
    print(f'identical to tempo_syl.pickle: {len(saved)} songs, three metrics'
          + ('' if same_order else '  [note] songs are in a different order there'))
else:
    print('no tempo_syl.pickle to compare against')

songs per bird x condition:
condition    heliox  regair
bird                       
s_b1349h_23     244     144
s_b1354_25       70     504
s_b1364_25      139     135
s_b1481h_24      28      62
s_b1484h_24      51      91
s_b1556h_24      21      79
identical to tempo_syl.pickle: 1568 songs, three metrics


In [58]:
# Cell W4: positions within the song, the Fig. 5D examples, checks, save
# position       index of the syllable within its bout (0 = first)
# position_norm  the same, divided by the bout's last index, so 0 = first and 1 = last syllable
ws_syl['position_norm'] = ws_syl.groupby('bout')['position'].transform(lambda x: x / x.max() if x.max() > 0 else 0.0)
shared = int((ws_syl.groupby('bout')['bird'].nunique() > 1).sum())
print(f'{ws_syl["bout"].nunique()} bouts in {ws_syl["file"].nunique()} wav files'
      + (f'   [CHECK] {shared} bout ids are used by more than one bird' if shared else ''))

# Same syllables, positions and labels as the table the original analysis worked from?
saved_path = os.path.join(DERIVED, 'syllable_df_for_session.pickle')
if os.path.exists(saved_path):
    saved = pd.read_pickle(saved_path)
    old = pd.DataFrame({'bird': saved['bird'].astype(str), 'condition': saved['condition'].astype(str),
                        'song_file': saved['file'].map(lambda p: os.path.basename(path_text(p))),
                        'position': saved['t_pos'].astype(float).fillna(-1), 'label': saved[LABCOL].astype(float)})
    new = ws_syl[['bird', 'condition', 'song_file', 'position', 'label']].astype({'position': float, 'label': float})
    new = new.assign(position=new['position'].fillna(-1))
    order = ['bird', 'condition', 'song_file', 'position', 'label']
    old, new = (d.sort_values(order).reset_index(drop=True) for d in (old, new))
    if len(old) != len(new):
        raise RuntimeError('syllable table differs from syllable_df_for_session.pickle. Rows per bird, here vs there:\n'
                           + pd.concat([new.groupby('bird').size().rename('here'),
                                        old.groupby('bird').size().rename('there')], axis=1).to_string())
    unequal = old.to_numpy() != new.to_numpy()
    if unequal.any():
        raise RuntimeError(f'syllable table differs from syllable_df_for_session.pickle in {int(unequal.any(axis=1).sum())} '
                           f'of {len(new)} rows; columns affected: {[c for c, bad in zip(order, unequal.any(axis=0)) if bad]}')
    print(f'identical to syllable_df_for_session.pickle: {len(new)} syllables (bird, condition, song, position, label)')
else:
    print('no syllable_df_for_session.pickle to compare against')

# ---- the two example songs of Fig. 5D: their syllable onsets ----------------------------------
example_rows = []
for which, name in TEMPO_EXAMPLES.items():
    if name is None:
        continue
    hit = ws_tempo[ws_tempo['song_file'] == os.path.basename(name)]
    if len(hit) != 1:
        raise KeyError(f'TEMPO_EXAMPLES[{which!r}] = {name!r} matches {len(hit)} songs in the tempo table')
    song = hit.iloc[0]
    onsets = np.sort(ws_syl.loc[ws_syl['file'] == song['file'], 'onset_s'].dropna().to_numpy())
    example_rows.append(pd.DataFrame({'example': which, 'bird': song['bird'], 'condition': song['condition'],
                                      'song_file': song['song_file'],
                                      'tempo_roughness_ms': song['tempo_roughness_ms'], 'onset_s': onsets}))
tempo_examples = (pd.concat(example_rows, ignore_index=True) if example_rows else
                  pd.DataFrame(columns=['example', 'bird', 'condition', 'song_file', 'tempo_roughness_ms', 'onset_s']))
if len(example_rows) < 2:
    print('[TO DO] TEMPO_EXAMPLES is not filled in, so Fig. 5D cannot be drawn yet. Set them in '
          'Cell W1, then rerun W1 and this cell.')

within_song = {
    'syllables': ws_syl[['bird', 'condition', 'song_file', 'position', 'position_norm', 'label']]
                 .astype({'song_file': 'category'}).reset_index(drop=True),
    'tempo_songs': ws_tempo[['bird', 'condition', 'song_file', 'song_len_s', 'n_syllables'] + TEMPO_COLS]
                   .reset_index(drop=True),
    'tempo_examples': tempo_examples,
}
for name in ('syllables', 'tempo_songs'):
    # all six starlings in both conditions, and no cluster paths in the table
    require_complete(within_song[name], name, 'bird', 'condition', species='starling')
for name, df in within_song.items():
    assert_portable(name, df)

within_song['meta'] = {
    'created': datetime.datetime.now().isoformat(timespec='seconds'),
    'source': 'one <bird>_syllable_df_augmented.pkl per starling',
    'settings': dict(MIN_SYLLABLES=MIN_SYLLABLES, MIN_SONG_LEN_S=MIN_SONG_LEN_S, TEMPO_SMOOTH_N=TEMPO_SMOOTH_N,
                     JUMP_THRESH_MS=JUMP_THRESH_MS, PAUSE_FACTOR=PAUSE_FACTOR, LABEL_COLUMN=LABCOL),
    'notes': {'label': 'syllable type from HDBSCAN on the UMAP embedding; -1 = unclustered',
              'tempo_songs': 'songs with enough syllables and duration; metrics from onset-to-onset intervals',
              'tempo_examples': 'syllable onsets (s) of the two songs drawn in Fig. 5D'},
    'versions': {'numpy': np.__version__, 'pandas': pd.__version__},
}
with open(WITHIN_SONG_DATA_PATH, 'wb') as f:
    pickle.dump(within_song, f, protocol=4)
print(f'\nsaved {WITHIN_SONG_DATA_PATH}  ({os.path.getsize(WITHIN_SONG_DATA_PATH) / 1e6:.1f} MB)')

2052 bouts in 2052 wav files
identical to syllable_df_for_session.pickle: 62090 syllables (bird, condition, song, position, label)
syllables: complete (62090 rows)
tempo_songs: complete (1568 rows)

saved /mnt/cube/lstanwic/umap_avgn_shared/lauren_umap/heliox_processing/data/derived/within_song_data.pkl  (2.5 MB)


In [59]:
# Cell L1: collect the latent-space tables (Fig. 5 F-H) for vae_latent_figures.ipynb
# These eight csv files are the saved results of the VAE analysis. This cell runs no
# analysis; it puts them in one pickle, keyed by file name, so the figures need one file.
LATENT_CSV_DIR = _path('HELIOX_LATENT_CSV_DIR', '/mnt/cube/lstanwic/umap_avgn_shared/lauren_umap/heliox_processing/notebooks/lo_vae/just_the_figures')   # <-- the folder holding the csv files
LATENT_DATA_PATH = _path('HELIOX_LATENT_DATA', os.path.join(DERIVED, 'vae_latent_data.pkl'))

LATENT_TABLES = ['strip_zf_r11g11_syl5_latents', 'strip_zf_r11g11_syl5_meta',   # 5F: one example syllable
                 'disp_heliox_birds', 'disp_heliox_stats',                      # 5G: latent drift
                 'gap_zebra_finch_pairs', 'gap_zebra_finch_pooled',             # 5H: movement toward regair
                 'gap_starling_pairs', 'gap_starling_pooled']

latent = {}
for name in LATENT_TABLES:
    latent[name] = pd.read_csv(os.path.join(LATENT_CSV_DIR, name + '.csv'))
    assert_portable(name, latent[name])
    print(f'  {name:32s} {len(latent[name]):>5d} rows  {latent[name].shape[1]:>2d} columns')
latent['meta'] = {'created': datetime.datetime.now().isoformat(timespec='seconds'),
                  'note': 'csv outputs of the VAE analysis, unchanged',
                  'versions': {'numpy': np.__version__, 'pandas': pd.__version__}}
with open(LATENT_DATA_PATH, 'wb') as f:
    pickle.dump(latent, f, protocol=4)
print(f'saved {LATENT_DATA_PATH}  ({os.path.getsize(LATENT_DATA_PATH) / 1e3:.0f} kB)')

  strip_zf_r11g11_syl5_latents       156 rows   4 columns
  strip_zf_r11g11_syl5_meta            1 rows  15 columns
  disp_heliox_birds                   11 rows   3 columns
  disp_heliox_stats                    1 rows   8 columns
  gap_zebra_finch_pairs               66 rows   3 columns
  gap_zebra_finch_pooled               1 rows  11 columns
  gap_starling_pairs                  36 rows   3 columns
  gap_starling_pooled                  1 rows  11 columns
saved /mnt/cube/lstanwic/umap_avgn_shared/lauren_umap/heliox_processing/data/derived/vae_latent_data.pkl  (14 kB)


In [60]:
# Cell E1: exposure-course data, settings and the ordering helper
# Feeds exposure_course_figures.ipynb: Fig. 5 I-M, Supp. Fig. S7, Supp. Table S1.
import gc
from scipy.stats import entropy

EXPOSURE_DATA_PATH = _path('HELIOX_EXPOSURE_DATA', os.path.join(DERIVED, 'exposure_course_data.pkl'))
BIOSOUND_CACHE_DIR = os.path.join(DERIVED, 'biosound_slim_day')
os.makedirs(BIOSOUND_CACHE_DIR, exist_ok=True)

MIN_N_PER_GROUP = 30      # syllables per bird x condition to enter the analysis at all
MIN_SONGS_PER_GROUP = 10  # songs per bird x condition to enter the analysis at all
MIN_SYL_PER_SONG = 12     # syllables in a song before its JSD is estimated

# date and half-hour file block are written into every starling wav name, e.g.
#   s_b1556h_24_2024-03-27_10-00-01-01_1312460_1322270_song_regair.wav
# and are the only reliable clock: start_time restarts inside each half-hour file.
WAV_NAME = re.compile(r'(?P<date>\d{4}-\d{2}-\d{2})_(?P<file_time_key>\d{2}-\d{2}-\d{2}-\d{2})_(?:(?P<start_sample>\d+)_\d+_)?')


def parse_wav_name(paths):
    """date, file_time_key and start_sample for each wav path. Stops if a name does not parse."""
    parts = paths.map(lambda p: os.path.basename(str(p))).str.extract(WAV_NAME)
    if parts['date'].isna().any():
        raise ValueError(f'{int(parts["date"].isna().sum())} wav names have no date in them')
    return parts


def order_within_bird(df, sort_key, min_n, keep, bird_col='bird', conditions=('regair', 'heliox')):
    """Put each bird's rows in recording order, one condition at a time, and add
       order_norm      0 to 1 across all of that bird's recording days in the condition
       day_number      1 on the first recording day
       order_norm_day  0 to 1 within the recording day
       n_day           rows that day
       first_day       True on day 1
    A bird x condition with fewer than min_n rows is left out and reported."""
    blocks = []
    for bird in sorted(df[bird_col].unique()):
        for cond in conditions:
            sub = df[(df[bird_col] == bird) & (df['condition'] == cond)].sort_values(sort_key)
            n = len(sub)
            if n < min_n:
                print(f'  [left out] {bird} {cond}: only {n} rows')
                continue
            out = sub[keep].copy()
            out['bird'], out['condition'] = bird, cond
            out['order_norm'] = np.arange(n) / (n - 1)
            day_of = {d: i + 1 for i, d in enumerate(pd.unique(sub['date']))}
            out['day_number'] = sub['date'].map(day_of).to_numpy()
            within = out.groupby('day_number')
            size = within['day_number'].transform('size')
            out['order_norm_day'] = (within.cumcount() / (size - 1).clip(lower=1)).to_numpy()
            out['n_day'] = size.to_numpy()
            out['first_day'] = out['day_number'] == 1
            blocks.append(out)
    return pd.concat(blocks, ignore_index=True)


print('exposure-course data ->', EXPOSURE_DATA_PATH)

exposure-course data -> /mnt/cube/lstanwic/umap_avgn_shared/lauren_umap/heliox_processing/data/derived/exposure_course_data.pkl


In [61]:
# Cell E2: syllable timing and amplitude (ISI, duration, amplitude)
# Amplitude is level_db from syllable_level_data.pkl (Cell A1), the RMS level of the syllable in the recorded
# audio. It replaces avg_power, the mean of each syllable's spectrogram image: the images were rescaled syllable
# by syllable, so that mean could not follow loudness. Spectral entropy now comes from BioSound (Cell E3).
FEATURES_A = ['duration', 'isi', 'level_db']
TABLE_COLUMNS = ['duration', 'isi']                    # read from the syllable tables; level_db is joined on
LEVEL_PATH = _path('HELIOX_SYLLABLE_LEVEL_DATA', os.path.join(DERIVED, 'syllable_level_data.pkl'))


def level_key(bird):
    """Starlings by number, since bird 1364 goes by two names; zebra finches by name."""
    return bird_num(bird) if str(bird).startswith('s_') else str(bird)


level = pd.read_pickle(LEVEL_PATH)
if isinstance(level, dict):          # in case the level table was saved inside a dict
    level = next(v for v in level.values() if isinstance(v, pd.DataFrame) and 'level_db' in v.columns)
level = pd.DataFrame({'key_': level['bird'].map(level_key).to_numpy(), 'row': level['row'].astype(int).to_numpy(),
                      'level_condition': level['condition'].astype(str).str.lower().to_numpy(),
                      'level_db': pd.to_numeric(level['level_db'], errors='coerce').to_numpy(float),
                      'clipped': level['clipped'].astype(bool).to_numpy()})


def add_level(df, species):
    """Joins level_db to every regair and heliox syllable on bird and row, where row is the syllable's position
    in its bird's own table (what syllable_level_data.pkl counts). Stops if any syllable finds no level, or
    finds it under another condition."""
    keep = df['condition'].isin(['regair', 'heliox'])
    t = df[keep].assign(key_=df.loc[keep, 'indv'].map(level_key))
    t = t.merge(level, on=['key_', 'row'], how='left', validate='one_to_one', indicator=True)
    unmatched = int((t['_merge'] != 'both').sum())
    disagree = int(((t['_merge'] == 'both') & (t['level_condition'] != t['condition'])).sum())
    if unmatched or disagree:
        raise ValueError(f'{species}: {unmatched} syllables have no row in syllable_level_data.pkl and {disagree} '
                         'have a different condition there')
    return t.drop(columns=['key_', 'level_condition', '_merge'])


def own_table(path, wanted):
    """A syllable table cut down to the wanted columns (first occurrence of each name), rows in file order."""
    raw = pd.read_pickle(path)
    cols = list(raw.columns)
    out = raw.iloc[:, [i for i, c in enumerate(cols) if c in wanted and c not in cols[:i]]].copy()
    del raw
    gc.collect()
    return out


# ---- starlings: one table for all six birds ----------------------------------------------
st = pd.read_pickle(os.path.join(DERIVED, 'all_birds_umap_clustered.pickle'))
if 'file_time_key' not in st.columns:
    st['file_time_key'] = parse_wav_name(st['key'])['file_time_key'].to_numpy()
if 'isi' not in st.columns:
    st['isi'] = st['time_between_syllables']

# The level table counts rows of each bird's own table, so every syllable of the combined table is first found
# in its bird's table, by the columns the two tables share. The old feature columns are matched too, which
# makes sure a match is the same syllable.
STARLING_TABLES = {
    's_b1349h_23': 's_b1349h_23/syllable_df_labels_mindistpoint2_featureanalysismod_20250612.pickle',
    's_b1354_25':  's_b1354_25/syllable_df_labels_mindistpoint2_featureanalysismod_20250611.pickle',
    's_b1364h_25': 's_b1364h_25/syllable_df_labels_mindistpoint2_featureanalysismod_20250612.pickle',
    's_b1481h_24': 's_b1481h_24/syllable_df_labels_mindistpoint2_featureanalysismod_20250612.pickle',
    's_b1484h_24': 's_b1484h_24/syllable_df_labels_mindistpoint2_featureanalysismod_20250612.pickle',
    's_b1556h_24': 's_b1556h_24/syllable_df_labels_mindistpoint2_featureanalysismod_20250612.pickle',
}
MATCH = ['key', 'start_time', 'end_time', 'syllables_sequence_id', 'syllables_sequence_pos',
         'duration', 'spectral_entropy', 'avg_power']


def match_value(s):
    # a wav name may be stored as text or as a Path; numbers are compared exactly
    return s.astype(str) if s.dtype == object else s


st['row'] = np.nan
for bird, rel in STARLING_TABLES.items():
    own = own_table(os.path.join(BASE_DIR, rel), MATCH + ['condition'])
    own['row'] = np.arange(len(own))
    shared = [c for c in MATCH if c in own.columns and c in st.columns]
    mine = st.index[(st['indv'].map(level_key) == level_key(bird)) & st['condition'].isin(['regair', 'heliox'])]
    own = own[own['condition'].astype(str).str.lower().isin(['regair', 'heliox'])]
    a = pd.DataFrame({c: match_value(st.loc[mine, c]) for c in shared}).assign(st_index=mine)
    b = pd.DataFrame({c: match_value(own[c]) for c in shared}).assign(row=own['row'].to_numpy())
    try:
        m = a.merge(b, on=shared, how='left', validate='one_to_one')
    except pd.errors.MergeError:
        raise ValueError(f'{bird}: the shared columns {shared} do not single out each syllable, so the combined '
                         'table cannot be matched to the bird table') from None
    if m['row'].isna().any():
        raise ValueError(f'{bird}: {int(m["row"].isna().sum())} of {len(m)} syllables of the combined table are not '
                         f'in {rel} (matched on {shared}).\n  combined table columns: {list(st.columns)}\n'
                         f'  bird table columns read: {list(own.columns)}')
    st.loc[m['st_index'].to_numpy(), 'row'] = m['row'].to_numpy()
    print(f'  {bird}: {len(m)} syllables found in the bird table, matched on {shared}')
    del own, a, b, m
    gc.collect()
st = add_level(st.astype({'row': float}).dropna(subset=['row']).astype({'row': int}), 'starling')
# day, then file within the day, then bout within the file, then syllable within the bout
ST_SORT = ['date', 'file_time_key', 'syllables_sequence_id', 'syllables_sequence_pos']
syllables_starling = order_within_bird(st, ST_SORT, MIN_N_PER_GROUP, FEATURES_A + ['clipped', 'date'],
                                       bird_col='indv')

# ---- zebra finches: one table per bird ---------------------------------------------------
ZF_FEATURE_FILES = {
    'z_r11g11_24': 'z_r11g11_24/syllable_df_labels_featureanalysismod_20250807.pickle',
    'z_p7y2_23':   'z_p7y2_23/syllable_df_labels_featureanalysismod_20250611.pickle',
    'z_y11c13_24': 'z_y11c13_24/syllable_df_labels_featureanalysismod_20250611.pickle',
    'z_c8c19_24':  'z_c8c19_24/syllable_df_labels_featureanalysismod_20250807.pickle',
    'z_p4g13_23':  'z_p4g13_23/syllable_df_labels_featureanalysismod_20250807.pickle',
    'z_g14g16_24': 'z_g14g16_24/syllable_df_labels_featureanalysismod_20250804.pickle',
}
ZF_SORT = ['date', 'hour', 'bout_index', 'on_ms']       # bout_index restarts each day


def standard_condition(c):
    s = str(c).lower()
    if 'heliox_fill' in s or 'transition' in s:
        return 'heliox_filling'                         # chamber still filling: not analysed
    if 'heliox' in s:
        return 'heliox'
    if 'regairb' in s:
        return 'regairb'                                # regular air after heliox: not analysed
    return 'regair' if ('regair' in s or 'regular' in s) else s


frames = []
for bird, rel in ZF_FEATURE_FILES.items():
    df = own_table(os.path.join(BASE_DIR, rel), ['condition', 'sess', 'hour', 'bout_index', 'on_ms'] + TABLE_COLUMNS)
    df['row'] = np.arange(len(df))                      # position in the bird's table, before anything is left out
    df['indv'], df['date'] = bird, df['sess']           # sess is the recording date, YYYY-MM-DD
    df['condition'] = df['condition'].map(standard_condition)
    frames.append(df)
zf = add_level(pd.concat(frames, ignore_index=True), 'zebra finch')
syllables_zebra_finch = order_within_bird(zf, ZF_SORT, MIN_N_PER_GROUP, FEATURES_A + ['clipped', 'date'],
                                          bird_col='indv')

del st, zf, frames, level
gc.collect()
for name, df in [('starling', syllables_starling), ('zebra finch', syllables_zebra_finch)]:
    print(f'\n{name}: {len(df)} syllables, {int(df["clipped"].sum())} of them flagged as clipped')
    print(df.groupby(['bird', 'condition']).size().unstack(fill_value=0).to_string())
    print('  median level_db by condition:', df.groupby('condition')['level_db'].median().round(2).to_dict())

  s_b1349h_23: 12683 syllables found in the bird table, matched on ['key', 'start_time', 'end_time', 'syllables_sequence_id', 'syllables_sequence_pos', 'duration', 'spectral_entropy', 'avg_power']
  s_b1354_25: 17338 syllables found in the bird table, matched on ['key', 'start_time', 'end_time', 'syllables_sequence_id', 'syllables_sequence_pos', 'duration', 'spectral_entropy', 'avg_power']
  s_b1364h_25: 49755 syllables found in the bird table, matched on ['key', 'start_time', 'end_time', 'syllables_sequence_id', 'syllables_sequence_pos', 'duration', 'spectral_entropy', 'avg_power']
  s_b1481h_24: 2336 syllables found in the bird table, matched on ['key', 'start_time', 'end_time', 'syllables_sequence_id', 'syllables_sequence_pos', 'duration', 'spectral_entropy', 'avg_power']
  s_b1484h_24: 6928 syllables found in the bird table, matched on ['key', 'start_time', 'end_time', 'syllables_sequence_id', 'syllables_sequence_pos', 'duration', 'spectral_entropy', 'avg_power']
  s_b1556h_24: 602

In [62]:
# Cell E3: BioSound features (first formant, mean F0, spectral mean, spectral entropy)
# The source tables run to ~10 GB each, so each bird is cut down once to the columns needed
# here and kept in biosound_slim_day/. Later runs read only those small files.
# Spectral entropy (entS, 0 to 1, from the syllable's power spectrum) replaces the old spectral_entropy column of
# the syllable tables, which was the entropy of the spectrogram averaged over frequency, so it measured how
# energy is spread over time, not over frequency. The slim files above were made before entS was needed, so
# entS is taken from the files of Cell A4 (biosound_paf_slim/), which hold every BioSound feature of the same
# tables in the same row order; if those are missing it is read from the source table once and kept.
BIOSOUND_FILES = {
    'starling': {
        's_b1349h_23': 's_b1349h_23/s-b1349-biosound-sylldf-2025-nonr.pickle',
        's_b1354_25':  's_b1354_25/s-b1354-biosound-sylldf-2025.pickle',
        's_b1364h_25': 's_b1364h_25/s-b1364-biosound-sylldf-2025.pickle',
        's_b1481h_24': 's_b1481h_24/biosound-features-biosound-df-2025.pickle',
        's_b1484h_24': 's_b1484h_24/s-b1484-biosound-sylldf-2025.pickle',
        's_b1556h_24': 's_b1556h_24/s-b1556-biosound-sylldf-2025.pickle'},
    'zebra finch': {
        'z_c8c19_24':  'z_c8c19_24/biosound-features-biosound-df-2025.pickle',
        'z_g14g16_24': 'z_g14g16_24/biosound-features-biosound-df-2025.pickle',
        'z_p4g13_23':  'z_p4g13_23/biosound-features-biosound-df-2025.pickle',
        'z_p7y2_23':   'z_p7y2_23/biosound-features-biosound-df-2025.pickle',
        'z_r11g11_24': 'z_r11g11_24/r11g11-biosound-sylldf-2025.pickle',
        'z_y11c13_24': 'z_y11c13_24/y11c13-biosound-sylldf-2025.pickle'},
}
FEATURES_B = ['F1', 'fund', 'meanS', 'entS']
SLIM_FEATURES = ['F1', 'fund', 'meanS']                 # the features the slim files were made with
SLIM_COLUMNS = set(SLIM_FEATURES) | {'condition', 'key', 'sess', 'date', 'bout_index', 'syllable_index',
                                     'syllables_sequence_id', 'syllables_sequence_pos', 'onsets_ms'}
BIO_SORT = {'starling': ['date', 'file_time_key', 'syllables_sequence_id', 'syllables_sequence_pos', 'onsets_ms'],
            'zebra finch': ['date', 'bout_index', 'syllable_index', 'onsets_ms']}
PAF_SLIM_DIR = os.path.join(DERIVED, 'biosound_paf_slim')     # written by Cell A4


def source_columns(rel, wanted):
    # by position, first occurrence of each name: a boolean column mask would copy every
    # array column of the 10 GB frame before narrowing it
    df = pd.read_pickle(os.path.join(BASE_DIR, rel))
    first = [i for i, c in enumerate(df.columns) if c in wanted and c not in list(df.columns[:i])]
    out = df.iloc[:, first].copy()
    del df
    gc.collect()
    return out


def load_biosound(bird, rel):
    slim_path = os.path.join(BIOSOUND_CACHE_DIR, f'{bird}_slim.pickle')
    if not os.path.exists(slim_path):
        slim = source_columns(rel, SLIM_COLUMNS)
        for f in SLIM_FEATURES:
            slim[f] = pd.to_numeric(slim[f], errors='coerce').astype('float32')
        slim['condition'] = np.where(slim['condition'].astype(str).str.lower().str.contains('heliox'),
                                     'heliox', 'regair')
        slim.to_pickle(slim_path)
    return pd.read_pickle(slim_path)


def load_entS(bird, rel, slim):
    """entS for every row of the slim table, checked to be the same syllables in the same order."""
    paf_path = os.path.join(PAF_SLIM_DIR, f'{bird}_paf.pickle')
    if os.path.exists(paf_path):
        paf = pd.read_pickle(paf_path)
    else:
        paf_path = os.path.join(BIOSOUND_CACHE_DIR, f'{bird}_entS.pickle')
        if not os.path.exists(paf_path):
            paf = source_columns(rel, {'condition', 'F1', 'entS'})
            for f in ['F1', 'entS']:
                paf[f] = pd.to_numeric(paf[f], errors='coerce').astype('float32')
            paf[['condition', 'F1', 'entS']].reset_index(drop=True).to_pickle(paf_path)
        paf = pd.read_pickle(paf_path)
    heliox = paf['condition'].astype(str).str.lower().str.contains('heliox').to_numpy()
    same = (len(paf) == len(slim) and (heliox == (slim['condition'] == 'heliox').to_numpy()).all()
            and np.array_equal(paf['F1'].to_numpy(np.float32), slim['F1'].to_numpy(np.float32), equal_nan=True))
    if not same:
        raise ValueError(f'{bird}: {os.path.basename(paf_path)} and the slim table are not the same syllables in the '
                         'same order (row count, condition or F1 differ), so entS cannot be attached')
    return paf['entS'].to_numpy()


biosound = {}
for species, files in BIOSOUND_FILES.items():
    frames = []
    for bird, rel in files.items():
        raw = load_biosound(bird, rel)
        raw['entS'] = load_entS(bird, rel, raw)
        raw['bird'] = bird
        if species == 'starling':
            parts = parse_wav_name(raw['key'])
            raw['date'], raw['file_time_key'] = parts['date'].to_numpy(), parts['file_time_key'].to_numpy()
        else:
            raw['date'] = raw['sess' if 'sess' in raw.columns else 'date'].astype(str)
        raw[FEATURES_B] = raw[FEATURES_B].astype(float)
        frames.append(raw)
    biosound[species] = order_within_bird(pd.concat(frames, ignore_index=True), BIO_SORT[species],
                                          MIN_N_PER_GROUP, FEATURES_B + ['date'])
    del frames
    gc.collect()
    print(f'\n{species}: {len(biosound[species])} syllables; recording days per bird x condition:')
    print(biosound[species].groupby(['bird', 'condition'])['day_number'].max().unstack(fill_value=0).to_string())
    print('  median entS by condition:', biosound[species].groupby('condition')['entS'].median().round(4).to_dict())


starling: 98569 syllables; recording days per bird x condition:
condition    heliox  regair
bird                       
s_b1349h_23       4       5
s_b1354_25        2       6
s_b1364h_25       4      15
s_b1481h_24       3       6
s_b1484h_24       2       3
s_b1556h_24       1       2
  median entS by condition: {'heliox': 0.6732, 'regair': 0.6624}

zebra finch: 68392 syllables; recording days per bird x condition:
condition    heliox  regair
bird                       
z_c8c19_24        2       2
z_g14g16_24       2       3
z_p4g13_23        2       2
z_p7y2_23         2       2
z_r11g11_24       2       2
z_y11c13_24       2       2
  median entS by condition: {'heliox': 0.7121, 'regair': 0.6894}


In [63]:
# Cell E4: per-song measures in starlings (tempo metrics, and JSD of syllable usage)
SONG_SORT = ['date', 'file_time_key', 'start_sample']    # start_sample orders songs within a file
TEMPO_METRICS = ['tempo_roughness_ms', 'tempo_jump_rate', 'tempo_pause_frac']

# ---- tempo: already one row per song -----------------------------------------------------
tempo = pd.read_pickle(os.path.join(DERIVED, 'tempo_syl.pickle'))
parts = parse_wav_name(tempo['wav_path'])
tempo['date'], tempo['file_time_key'] = parts['date'].to_numpy(), parts['file_time_key'].to_numpy()
tempo['start_sample'] = parts['start_sample'].astype(int).to_numpy()
tempo['song_file'] = tempo['wav_path'].map(lambda p: os.path.basename(str(p)))
songs_tempo = order_within_bird(tempo, SONG_SORT, MIN_SONGS_PER_GROUP, TEMPO_METRICS + ['song_file', 'date'],
                                conditions=('heliox', 'regair'))   # block order of the original analysis

# ---- JSD: each song's syllable usage against that bird's own regular air usage -----------
syl = pd.read_pickle(os.path.join(DERIVED, 'syllable_df_for_session.pickle'))
syl = syl[syl[LABCOL] >= 0]                                # drop HDBSCAN noise


def jsd(counts_a, counts_b):
    labels = sorted(set(counts_a.index) | set(counts_b.index))
    pa = counts_a.reindex(labels, fill_value=0).to_numpy(dtype=float)
    pb = counts_b.reindex(labels, fill_value=0).to_numpy(dtype=float)
    if pa.sum() <= 0 or pb.sum() <= 0:
        return np.nan
    pa, pb = pa / pa.sum(), pb / pb.sum()
    m = 0.5 * (pa + pb)
    return 0.5 * (entropy(pa + 1e-12, m + 1e-12) + entropy(pb + 1e-12, m + 1e-12))


rows = []
for bird, dfb in syl.groupby('bird'):
    baseline = dfb.loc[dfb['condition'] == 'regair', LABCOL].value_counts()
    if baseline.sum() < 50:
        print(f'  [left out] {bird}: only {baseline.sum()} regair syllables for a baseline')
        continue
    for song_file, dfs in dfb.groupby('file'):
        if len(dfs) < MIN_SYL_PER_SONG:
            continue
        cond = dfs['condition'].mode().iat[0]
        counts = dfs[LABCOL].value_counts()
        # A regair song is scored against the baseline WITHOUT itself. Leaving it in would
        # pull regair JSD down and manufacture a condition difference.
        ref = baseline.subtract(counts, fill_value=0).clip(lower=0) if cond == 'regair' else baseline
        if ref.sum() < 50:
            continue
        rows.append(dict(bird=bird, wav_path=song_file, condition=cond,
                         jsd_song=jsd(counts, ref), n_syllables=len(dfs)))
jsd_songs = pd.DataFrame(rows)
jsd_songs = jsd_songs[np.isfinite(jsd_songs['jsd_song'])]
parts = parse_wav_name(jsd_songs['wav_path'])
jsd_songs['date'], jsd_songs['file_time_key'] = parts['date'].to_numpy(), parts['file_time_key'].to_numpy()
jsd_songs['start_sample'] = parts['start_sample'].astype(int).to_numpy()
jsd_songs['song_file'] = jsd_songs['wav_path'].map(lambda p: os.path.basename(str(p)))
songs_jsd = order_within_bird(jsd_songs, SONG_SORT, MIN_SONGS_PER_GROUP,
                              ['jsd_song', 'n_syllables', 'song_file', 'date'], conditions=('heliox', 'regair'))

print('songs per bird x condition (tempo | JSD):')
print(pd.concat([songs_tempo.groupby(['bird', 'condition']).size().rename('tempo'),
                 songs_jsd.groupby(['bird', 'condition']).size().rename('jsd')], axis=1).to_string())

songs per bird x condition (tempo | JSD):
                       tempo  jsd
bird        condition            
s_b1349h_23 heliox       244  195
            regair       144   99
s_b1354_25  heliox        70   68
            regair       504  491
s_b1364_25  heliox       139  127
            regair       135  119
s_b1481h_24 heliox        28   18
            regair        62   58
s_b1484h_24 heliox        51   52
            regair        91   91
s_b1556h_24 heliox        21   21
            regair        79   79


In [64]:
# Cell E5: check and save the exposure-course data
exposure = {
    'syllables_starling': syllables_starling, 'syllables_zebra_finch': syllables_zebra_finch,
    'biosound_starling': biosound['starling'], 'biosound_zebra_finch': biosound['zebra finch'],
    'songs_tempo': songs_tempo, 'songs_jsd': songs_jsd,
}
for name, df in exposure.items():
    # all six birds of the species, in both conditions, with no cluster paths in the table
    require_complete(df, name, 'bird', 'condition',
                     species='zebra_finch' if name.endswith('zebra_finch') else 'starling')
    assert_portable(name, df)

exposure['meta'] = {
    'created': datetime.datetime.now().isoformat(timespec='seconds'),
    'thresholds': dict(MIN_N_PER_GROUP=MIN_N_PER_GROUP, MIN_SONGS_PER_GROUP=MIN_SONGS_PER_GROUP,
                       MIN_SYL_PER_SONG=MIN_SYL_PER_SONG, LABEL_COLUMN=LABCOL),
    'units': {'isi, duration': 's', 'level_db': 'dB re full scale', 'F1, fund, meanS': 'Hz', 'entS': '0 to 1', 'tempo_roughness_ms': 'ms',
              'jsd_song': "JSD of a song's syllable usage from the bird's own regular air usage"},
    'bird_names': 's_b1364h_25 (syllable tables) and s_b1364_25 (song tables) are the same bird',
    'versions': {'numpy': np.__version__, 'pandas': pd.__version__},
}
with open(EXPOSURE_DATA_PATH, 'wb') as f:
    pickle.dump(exposure, f, protocol=4)
print(f'\nsaved {EXPOSURE_DATA_PATH}  ({os.path.getsize(EXPOSURE_DATA_PATH) / 1e6:.1f} MB)')

syllables_starling: complete (95063 rows)
syllables_zebra_finch: complete (63728 rows)
biosound_starling: complete (98569 rows)
biosound_zebra_finch: complete (68392 rows)
songs_tempo: complete (1568 rows)
songs_jsd: complete (1418 rows)

saved /mnt/cube/lstanwic/umap_avgn_shared/lauren_umap/heliox_processing/data/derived/exposure_course_data.pkl  (26.9 MB)


## Supplementary Figs. S1, S3 and S4

In [65]:
# Cell C1: clustering validation (Supp. Fig. S1), settings and each starling's UMAP coordinates
# Feeds clustering_validation_figures.ipynb.
import math
import hdbscan
from joblib import Parallel, delayed
from importlib.metadata import version as package_version

CLUSTERING_DATA_PATH = _path('HELIOX_CLUSTERING_DATA', os.path.join(DERIVED, 'clustering_validation_data.pkl'))

# Bird 1364 enters from its condition-balanced table (the 14,069 syllables in the figure legend),
# under the name the figure prints for it. The other five use the tables in Cell F1.
S1_TABLES = dict(STARLING_FEATURE_FILES, s_b1364h_25='s_b1364_25/syllable_df_labels_mindist2_20251203.pickle')

CHOSEN_PERCENT = 0.5                                  # the threshold used in the paper
SWEEP_PERCENTS = [0.1, 0.2, 0.5, 1, 5, 10, 50, 100]   # C-E: min_cluster_size as % of the bird's syllables
SUBSAMPLE_MCS = [0.001, 0.0025, 0.005, 0.01, 0.05]    # A-B: min_cluster_size as a fraction of the subsample
SUBSAMPLE_FRACS = np.linspace(0.10, 1.00, 12)         # A-B: share of the bird's syllables in a subsample
N_DRAWS = 5                                           # A-B: subsamples drawn per bird and share
MIN_SUBSAMPLE = 400                                   # A-B: a subsample smaller than this is not clustered
SUBSAMPLE_SEED = 123
N_JOBS = 8                                            # parallel HDBSCAN runs in Cell C2

c_umap = {}
for bird in sorted(S1_TABLES):                        # sorted: the subsample draws below depend on bird order
    df = read_columns(os.path.join(BASE_DIR, S1_TABLES[bird]), ['condition', 'umap', 'umap_x', 'umap_y'])
    df['condition'] = df['condition'].astype(str).str.lower()
    df = df[df['condition'].isin(COND_ORDER)]         # drops 'regairb' (regular air recorded after heliox)
    x, y = umap_xy(df)
    c_umap[bird] = pd.DataFrame({'bird': bird, 'condition': df['condition'].to_numpy(), 'umap_x': x, 'umap_y': y})

c_all = pd.concat(c_umap.values(), ignore_index=True)
require_complete(c_all, 'S1 UMAP coordinates', 'bird', 'condition', species='starling')
if not np.isfinite(c_all[['umap_x', 'umap_y']].to_numpy()).all():
    raise ValueError('some UMAP coordinates are missing')
counts = c_all.groupby(['bird', 'condition']).size().unstack(fill_value=0)
print(counts.assign(total=counts.sum(axis=1)).to_string())

S1 UMAP coordinates: complete (59377 rows)
condition    heliox  regair  total
bird                              
s_b1349h_23    8232    4451  12683
s_b1354_25     2238   15100  17338
s_b1364h_25    6671    7398  14069
s_b1481h_24     696    1640   2336
s_b1484h_24    2502    4426   6928
s_b1556h_24    1233    4790   6023


In [66]:
# Cell C2: HDBSCAN threshold sweep (Supp. S1 C-E) and subsampling (Supp. S1 A-B)
# Takes a while: about 1,800 clustering runs. The two largest thresholds (50% and 100%) ask HDBSCAN
# for a neighbour table of roughly N x N numbers, a few GB for the largest bird, so the sweep runs
# one fit at a time and only the small subsample fits run in parallel.
def cluster_stats(Y, min_cluster_size):
    """Clusters found and share of syllables left unclustered in one HDBSCAN run."""
    labels = hdbscan.HDBSCAN(min_cluster_size=int(min_cluster_size), core_dist_n_jobs=1,
                             approx_min_span_tree=False).fit_predict(Y)
    return int(np.unique(labels[labels >= 0]).size), float((labels < 0).mean())


# ---- C-E: all of each bird's syllables, clustered at each threshold --------------------------
rows = []
for bird, d in c_umap.items():
    Y = d[['umap_x', 'umap_y']].to_numpy()
    for percent in SWEEP_PERCENTS:
        mcs = max(2, int(math.ceil(percent / 100.0 * len(Y))))       # rounded up, as in the figure
        n_clusters, noise = cluster_stats(Y, mcs)
        rows.append((bird, percent, len(Y), mcs, n_clusters, noise))
c_sweep = pd.DataFrame(rows, columns=['bird', 'percent', 'n_syllables', 'min_cluster_size', 'n_clusters', 'noise_frac'])

# ---- A-B: random subsamples of each bird, clustered at five thresholds ------------------------
# The draws are made first, one after another, so that they do not depend on how the fits are
# scheduled; the fits themselves use no random numbers.
rng = np.random.default_rng(SUBSAMPLE_SEED)
draws = []
for bird, d in c_umap.items():
    Y = d[['umap_x', 'umap_y']].to_numpy()
    for share in SUBSAMPLE_FRACS:
        n = int(np.floor(share * len(Y)))
        if n < MIN_SUBSAMPLE:
            print(f'  [left out] {bird} at {share * 100:.0f}%: {n} syllables is under {MIN_SUBSAMPLE}')
            continue
        for draw in range(N_DRAWS):
            draws.append((bird, float(share), draw, Y[rng.choice(len(Y), size=n, replace=False)]))


def cluster_one_draw(bird, share, draw, Y):
    out = []
    for frac in SUBSAMPLE_MCS:
        mcs = max(2, int(frac * len(Y)))                             # rounded down, as in the figure
        n_clusters, noise = cluster_stats(Y, mcs)
        out.append((bird, share, draw, frac, len(Y), mcs, n_clusters, noise))
    return out


blocks = Parallel(n_jobs=N_JOBS)(delayed(cluster_one_draw)(*d) for d in draws)
c_subsample = pd.DataFrame([row for block in blocks for row in block],
                           columns=['bird', 'sample_frac', 'draw', 'mcs_frac', 'n_sample', 'min_cluster_size',
                                    'n_clusters', 'noise_frac'])
del draws, blocks
gc.collect()

print(f'sweep: {len(c_sweep)} runs; subsampling: {len(c_subsample)} runs\n\nclusters per bird at each threshold (%):')
print(c_sweep.pivot(index='bird', columns='percent', values='n_clusters').to_string())

  [left out] s_b1481h_24 at 10%: 233 syllables is under 400
sweep: 48 runs; subsampling: 1775 runs

clusters per bird at each threshold (%):
percent      0.1    0.2    0.5    1.0    5.0    10.0   50.0   100.0
bird                                                               
s_b1349h_23    221    133     42     22      2      0      0      0
s_b1354_25     164    114     52     24      2      0      0      0
s_b1364h_25    286    164     42      3      0      0      0      0
s_b1481h_24    173    104     52     22      2      0      0      0
s_b1484h_24    168    108     42     22      7      2      0      0
s_b1556h_24    215    142     59     27      2      0      0      0


In [67]:
# Cell C3: check and save the clustering validation data
chosen = c_sweep[np.isclose(c_sweep['percent'], CHOSEN_PERCENT)].set_index('bird')
summary = pd.DataFrame({'syllables': chosen['n_syllables'], 'clusters': chosen['n_clusters'],
                        'noise_%': (chosen['noise_frac'] * 100).round(1)})

# Does this figure describe the clustering the other analyses use? Compared with the labels in the
# within-song data (Cell W4), which come from the augmented tables. Birds are matched by number,
# since bird 1364 goes by two names.
ws_path = _path('HELIOX_WITHIN_SONG_DATA', os.path.join(DERIVED, 'within_song_data.pkl'))
if os.path.exists(ws_path):
    used = pd.read_pickle(ws_path)['syllables']
    used = used.assign(num=used['bird'].map(bird_num)).groupby('num')['label'].agg(
        syllables='size', clusters=lambda s: s[s >= 0].nunique(), noise=lambda s: (s < 0).mean() * 100)
    num = summary.index.map(bird_num)
    summary['syllables_used'] = used['syllables'].reindex(num).to_numpy()
    summary['clusters_used'] = used['clusters'].reindex(num).to_numpy()
    summary['noise_%_used'] = used['noise'].reindex(num).round(1).to_numpy()
    summary['same'] = np.where((summary['syllables'] == summary['syllables_used'])
                               & (summary['clusters'] == summary['clusters_used']), '', '[CHECK]')
    print(f'at {CHOSEN_PERCENT}%: this figure, and (_used) the {LABCOL} labels the other analyses use')
else:
    print(f'at {CHOSEN_PERCENT}% (no within-song data at {ws_path} to compare the labels with)')
print(summary.to_string())

clustering = {'sweep': c_sweep, 'subsampling': c_subsample}
for name, df in clustering.items():
    n_birds = df['bird'].nunique()
    if n_birds != EXPECTED_BIRDS['starling']:
        message = f'{name} holds {n_birds} birds, expected {EXPECTED_BIRDS["starling"]}'
        if not ALLOW_INCOMPLETE:
            raise RuntimeError(message + '. Set ALLOW_INCOMPLETE = True in Cell 2 to continue anyway.')
        INCOMPLETE.append(message)
        print('[INCOMPLETE]', message)
    assert_portable(name, df)

clustering['meta'] = {
    'created': datetime.datetime.now().isoformat(timespec='seconds'),
    'tables': {bird: os.path.basename(rel) for bird, rel in S1_TABLES.items()},
    'settings': dict(CHOSEN_PERCENT=CHOSEN_PERCENT, SWEEP_PERCENTS=SWEEP_PERCENTS, SUBSAMPLE_MCS=SUBSAMPLE_MCS,
                     SUBSAMPLE_FRACS=[float(s) for s in SUBSAMPLE_FRACS], N_DRAWS=N_DRAWS,
                     MIN_SUBSAMPLE=MIN_SUBSAMPLE, SUBSAMPLE_SEED=SUBSAMPLE_SEED),
    'notes': {'sweep': "HDBSCAN on all of a bird's UMAP-embedded syllables; min_cluster_size = percent of them, "
                       'rounded up',
              'subsampling': 'HDBSCAN on a random subsample (no replacement); min_cluster_size = mcs_frac of the '
                             'subsample, rounded down',
              'noise_frac': 'share of syllables HDBSCAN leaves unclustered',
              's_b1364h_25': 'from the condition-balanced table of this bird (folder s_b1364_25)'},
    # cluster counts at the smallest thresholds can differ between hdbscan / scikit-learn versions
    'versions': {'numpy': np.__version__, 'pandas': pd.__version__, 'hdbscan': package_version('hdbscan'),
                 'scikit-learn': package_version('scikit-learn')},
}
with open(CLUSTERING_DATA_PATH, 'wb') as f:
    pickle.dump(clustering, f, protocol=4)
print(f'\nsaved {CLUSTERING_DATA_PATH}  ({os.path.getsize(CLUSTERING_DATA_PATH) / 1e3:.0f} kB)')

at 0.5%: this figure, and (_used) the hdbscan_labels_new_0_005 labels the other analyses use
             syllables  clusters  noise_%  syllables_used  clusters_used  noise_%_used     same
bird                                                                                           
s_b1349h_23      12683        42     25.6           12683             47          26.7  [CHECK]
s_b1354_25       17338        52     19.3           20051             41          10.1  [CHECK]
s_b1364h_25      14069        42     46.6           14069             34          27.8  [CHECK]
s_b1481h_24       2336        52     21.2            2336             55          22.2  [CHECK]
s_b1484h_24       6928        42      8.7            6928             48          10.7  [CHECK]
s_b1556h_24       6023        59     14.0            6023             60          13.3  [CHECK]

saved /mnt/cube/lstanwic/umap_avgn_shared/lauren_umap/heliox_processing/data/derived/clustering_validation_data.pkl  (115 kB)


In [68]:
# Cell M1: mutual information (Supp. Fig. S3), for mutual_information_figures.ipynb
# Needs Cells 1-4 and F1. This cell runs no bootstrap. It puts the two bundles the MI bootstrap
# notebook saved into one pickle: each bird's 1,000 MI draws per condition, its null draws and its P.
MI_DATA_PATH = _path('HELIOX_MI_DATA', os.path.join(DERIVED, 'mutual_information_data.pkl'))
MI_DIR = _path('HELIOX_MI_DIR', os.path.join(BASE_DIR, 'MI'))
MI_BUNDLES = {
    'starling': 'ALLSTARLINGS_MI_starling_bootstraps.pkl',      # regular air recorded after heliox left out
    'zebra_finch': 'ALLZEBRAFINCHES_MI_motif0_bootstraps.pkl',  # motif syllables only, no gap allowed between motifs
}
NULL_KEYS = ('null', 'null_delta', 'dmi_null')                  # names the null has had in different versions

bird_rows, draw_frames, null_frames, mi_settings = [], [], [], {}
for species, file_name in MI_BUNDLES.items():
    bundle = load_pickle(os.path.join(MI_DIR, file_name))
    mi_df, dists = bundle['mi_df'], bundle['mi_dists']
    # the bundle's own settings, minus the tables and anything that is a cluster path
    mi_settings[species] = {k: py(v) for k, v in bundle.items() if isinstance(v, (str, int, float, np.number))
                            and not (isinstance(v, str) and os.sep in v)}
    for _, row in mi_df.iterrows():
        bird = str(row['subject'])
        pack = dists[bird]
        r, h = np.asarray(pack['mi_R'], float), np.asarray(pack['mi_H'], float)
        # The plotting code pairs draw k of regular air with draw k of heliox; that pairing is only
        # well defined when both conditions have the same number of draws and none is missing.
        if len(r) != len(h) or not (np.isfinite(r).all() and np.isfinite(h).all()):
            raise ValueError(f'{bird}: {len(r)} regair and {len(h)} heliox draws, or draws that are not numbers')
        null_key = next((k for k in NULL_KEYS if pack.get(k) is not None), None)
        null = np.asarray(pack[null_key], float) if null_key else np.array([])
        delta = r - h
        bird_rows.append(dict(species=species, bird=bird, mi_regair=r.mean(), mi_heliox=h.mean(),
                              dmi=delta.mean(), dmi_lo=np.percentile(delta, 2.5), dmi_hi=np.percentile(delta, 97.5),
                              p=float(row['p']), n_boot=len(r), n_null=len(null),
                              dmi_saved=float(row['dMI']) if 'dMI' in row else np.nan))
        draw_frames += [pd.DataFrame({'species': species, 'bird': bird, 'condition': cond, 'draw': np.arange(len(v)),
                                      'mi': v}) for cond, v in (('regair', r), ('heliox', h))]
        null_frames.append(pd.DataFrame({'species': species, 'bird': bird, 'draw': np.arange(len(null)),
                                         'dmi_null': null}))

mi_birds = pd.DataFrame(bird_rows)
mi_draws = pd.concat(draw_frames, ignore_index=True)
mi_null = pd.concat(null_frames, ignore_index=True)

# The draws should give back the dMI the bundle stored; if not, the bundle and its draws disagree.
gap = np.nanmax(np.abs(mi_birds['dmi'] - mi_birds['dmi_saved'])) if mi_birds['dmi_saved'].notna().any() else np.nan
print(f'largest difference between dMI from the draws and the stored dMI: {gap:.2g}'
      + ('   [CHECK]' if np.isfinite(gap) and gap > 1e-9 else ''))
# The stored P is a count of null draws at least as far from zero as dMI; this says whether that
# count, (k + 1) / (n + 1), gives back the stored values. A difference means another rule was used.
if len(mi_null):
    k = mi_null.merge(mi_birds[['bird', 'dmi']], on='bird').assign(beyond=lambda x: x['dmi_null'].abs() >= x['dmi'].abs()) \
               .groupby('bird')['beyond'].sum()
    p_count = (k.reindex(mi_birds['bird']).to_numpy() + 1) / (mi_birds['n_null'].to_numpy() + 1)
    print(f'stored P vs (k + 1) / (n + 1) from the null draws: largest difference {np.max(np.abs(p_count - mi_birds["p"])):.2g}')
else:
    print('[note] the bundles hold no null draws; P is taken as stored')
print(mi_birds.drop(columns='dmi_saved').round(4).to_string(index=False))
for species, g in mi_birds.groupby('species'):
    print(f'{species}: P < 0.05 in {int((g["p"] < 0.05).sum())} of {len(g)} birds')

mutual_information = {'birds': mi_birds.drop(columns='dmi_saved'), 'bootstrap': mi_draws, 'null': mi_null}
require_complete(mi_draws, 'bootstrap', 'bird', 'condition', species_col='species')
for name, df in mutual_information.items():
    assert_portable(name, df)

mutual_information['meta'] = {
    'created': datetime.datetime.now().isoformat(timespec='seconds'),
    'bundles': MI_BUNDLES,
    'bundle_settings': mi_settings,
    'notes': {'bootstrap': 'MI (bits) of each bootstrap draw: ~50% of the bird\'s bouts in that condition, '
                           'without replacement; draw k of regair is paired with draw k of heliox',
              'birds': 'mi_regair, mi_heliox: mean of the draws; dmi: mean of regair - heliox over paired draws; '
                       'dmi_lo, dmi_hi: 2.5 and 97.5 percentiles; p: from the within-condition null, as stored',
              'null': 'MI difference between two random halves of the regular air bouts'},
    'versions': {'numpy': np.__version__, 'pandas': pd.__version__},
}
with open(MI_DATA_PATH, 'wb') as f:
    pickle.dump(mutual_information, f, protocol=4)
print(f'\nsaved {MI_DATA_PATH}  ({os.path.getsize(MI_DATA_PATH) / 1e6:.1f} MB)')

largest difference between dMI from the draws and the stored dMI: 6e-16
stored P vs (k + 1) / (n + 1) from the null draws: largest difference 0
    species        bird  mi_regair  mi_heliox     dmi  dmi_lo  dmi_hi      p  n_boot  n_null
   starling s_b1349h_23     2.4698     2.2797  0.1901  0.0924  0.2890 0.0190    1000    1000
   starling  s_b1354_25     2.1550     2.0547  0.1004 -0.0521  0.2530 0.0480    1000    1000
   starling s_b1364h_25     2.3982     2.2213  0.1769  0.0827  0.2802 0.0010    1000    1000
   starling s_b1481h_24     3.4228     3.3106  0.1122 -0.1490  0.3930 0.3367    1000    1000
   starling s_b1484h_24     2.5045     2.5909 -0.0864 -0.2088  0.0240 0.1988    1000    1000
   starling s_b1556h_24     2.7927     3.1554 -0.3627 -0.5782 -0.1513 0.0010    1000    1000
zebra_finch  z_c8c19_24     1.7976     1.7635  0.0341  0.0029  0.0647 0.0340    1000    1000
zebra_finch z_g14g16_24     1.5850     1.5850  0.0000 -0.0000  0.0000 0.3127    1000    1000
zebra_finch  z_p4g1

In [69]:
# Cell R1: syllable repetition in starlings (Supp. Fig. S4), for repetition_figures.ipynb
# Needs Cells 1-4 and F1. Reads the same six augmented tables as Cell W2 and rebuilds the two tables
# the original repeat analysis made: one row per pair of neighbouring syllables, one row per run.
REPETITION_DATA_PATH = _path('HELIOX_REPETITION_DATA', os.path.join(DERIVED, 'repetition_data.pkl'))

t_frames, r_frames = [], []
for bird in STARLING_BIRDS:
    df = read_columns(os.path.join(AUGMENTED_DIR, bird, f'{bird}_syllable_df_augmented.pkl'),
                      ['file', BOUTCOL, 'condition', 'onsets_ms', 'offsets_ms', LABCOL])
    d = pd.DataFrame({
        'condition': df['condition'].astype(str).str.lower().to_numpy(),
        'bout': df[BOUTCOL if BOUTCOL in df.columns else 'file'].map(str).to_numpy(),   # bout_index holds the wav path
        'onset_s': pd.to_numeric(df['onsets_ms'], errors='coerce').to_numpy(),    # misnamed: these columns hold
        'offset_s': pd.to_numeric(df['offsets_ms'], errors='coerce').to_numpy(),  # seconds in the starling tables
        'label': pd.to_numeric(df[LABCOL], errors='coerce').to_numpy()})
    del df
    other = sorted(set(d['condition']) - set(COND_ORDER))
    if other:
        raise ValueError(f'{bird}: conditions other than regair and heliox: {other}')
    if d['offset_s'].median() > 600:
        raise ValueError(f'{bird}: onsets_ms / offsets_ms look like milliseconds here; these tables should hold seconds')

    # Unclustered syllables are dropped BEFORE pairing, as in the original analysis, so a syllable's
    # neighbour is the next clustered syllable and an ISI can span an unclustered one.
    d = d[d['label'] >= 0]
    # condition, then bout, then onset: the order the original loops visited them in (regair first)
    d = d.assign(condition=pd.Categorical(d['condition'], COND_ORDER)) \
         .sort_values(['condition', 'bout', 'onset_s'], kind='mergesort').reset_index(drop=True)
    ties = int(d.duplicated(['bout', 'onset_s']).sum())
    if ties:
        print(f'  [note] {bird}: {ties} syllables share an onset with another in the same bout; their order is the table order')

    group = d.groupby(['condition', 'bout'], sort=False, observed=True).ngroup().to_numpy()
    cond, lab = d['condition'].astype(str).to_numpy(), d['label'].to_numpy(int)
    on, off = d['onset_s'].to_numpy(float), d['offset_s'].to_numpy(float)
    same_bout = group[1:] == group[:-1]          # row i and row i + 1 are neighbours in one bout

    # ---- transitions: syllable i and the next one ----------------------------------------------
    i = np.flatnonzero(same_bout)
    t_frames.append(pd.DataFrame({
        'bird': bird, 'condition': cond[i], 'from_label': lab[i], 'to_label': lab[i + 1],
        'is_repeat': lab[i + 1] == lab[i],
        'duration_s': off[i] - on[i],            # duration of the first syllable of the pair
        'isi_s': on[i + 1] - off[i]}))           # silence between the two

    # ---- runs: stretches of one syllable type within a bout --------------------------------------
    starts = np.r_[True, ~same_bout | (lab[1:] != lab[:-1])]
    first = np.flatnonzero(starts)
    r_frames.append(pd.DataFrame({
        'bird': bird, 'condition': cond[first], 'label': lab[first],
        'run_length': np.diff(np.r_[first, len(lab)]),
        # clustered syllables in the run's bout: the per-bird summaries leave out one-syllable bouts
        'bout_syllables': np.bincount(group)[group[first]]}))

rep_transitions = pd.concat(t_frames, ignore_index=True)
rep_runs = pd.concat(r_frames, ignore_index=True)
del t_frames, r_frames, d
gc.collect()

# Same rows as the tables the original analysis saved? Checked when those csv files exist.
for name, new, cols in [
        ('transition', rep_transitions.rename(columns={'from_label': 'from', 'to_label': 'to'}),
         ['indv', 'condition', 'from', 'to', 'is_repeat', 'duration_s', 'isi_s']),
        ('run', rep_runs, ['indv', 'condition', 'label', 'run_length'])]:
    saved_path = os.path.join(STARLING_SYL_DIR, f'ALL_BIRDS_{name}_level_table.csv')
    if not os.path.exists(saved_path):
        print(f'no {os.path.basename(saved_path)} to compare against')
        continue
    old = pd.read_csv(saved_path)[cols]
    new = new.rename(columns={'bird': 'indv'})[cols]
    if 'is_repeat' in cols:
        new = new.assign(is_repeat=new['is_repeat'].astype(int))         # saved as 0 / 1
    # Times rounded to the microsecond before sorting. Read back from csv, a time can be 1 ulp off, and
    # sorting on the raw values then puts tied rows (same labels, same duration) in a different order.
    times = {c: 6 for c in ('duration_s', 'isi_s') if c in cols}
    old, new = (x.round(times).sort_values(cols).reset_index(drop=True) for x in (old, new))
    if len(old) != len(new):
        raise RuntimeError(f'{name} table differs from {os.path.basename(saved_path)}. Rows per bird x condition, '
                           'here vs there:\n' + pd.concat([new.groupby(['indv', 'condition']).size().rename('here'),
                                                           old.groupby(['indv', 'condition']).size().rename('there')],
                                                          axis=1).to_string())
    text = [c for c in cols if c in ('indv', 'condition')]
    numbers = [c for c in cols if c not in text]
    if (old[text].astype(str).to_numpy() != new[text].astype(str).to_numpy()).any() or \
            not np.allclose(old[numbers].to_numpy(float), new[numbers].to_numpy(float), atol=1e-6, equal_nan=True):
        raise RuntimeError(f'{name} table has the same number of rows as {os.path.basename(saved_path)} '
                           'but different values')
    print(f'identical to {os.path.basename(saved_path)}: {len(new)} rows')

summary = rep_transitions.groupby(['bird', 'condition']).agg(transitions=('is_repeat', 'size'),
                                                            repeat_share=('is_repeat', 'mean'))
summary['runs'] = rep_runs.groupby(['bird', 'condition']).size()
print(summary.round(3).to_string())

repetition = {'transitions': rep_transitions, 'runs': rep_runs}
for name, df in repetition.items():
    # all six starlings in both conditions, and no cluster paths in the table
    require_complete(df, name, 'bird', 'condition', species='starling')
    assert_portable(name, df)

repetition['meta'] = {
    'created': datetime.datetime.now().isoformat(timespec='seconds'),
    'source': 'one <bird>_syllable_df_augmented.pkl per starling, as in Cell W2',
    'settings': dict(LABEL_COLUMN=LABCOL, unclustered='dropped before pairing neighbours'),
    'notes': {'transitions': 'neighbouring clustered syllables of one bout; is_repeat = same label',
              'runs': 'stretches of one label within a bout; bout_syllables = clustered syllables in that bout',
              'bird_names': 's_b1364_25 here is s_b1364h_25 in the syllable-feature tables'},
    'units': {'duration_s, isi_s': 's'},
    'versions': {'numpy': np.__version__, 'pandas': pd.__version__},
}
with open(REPETITION_DATA_PATH, 'wb') as f:
    pickle.dump(repetition, f, protocol=4)
print(f'\nsaved {REPETITION_DATA_PATH}  ({os.path.getsize(REPETITION_DATA_PATH) / 1e6:.1f} MB)')

identical to ALL_BIRDS_transition_level_table.csv: 48653 rows
identical to ALL_BIRDS_run_level_table.csv: 37343 rows
                       transitions  repeat_share   runs
bird        condition                                  
s_b1349h_23 heliox            5888         0.180   5215
            regair            2799         0.241   2342
s_b1354_25  heliox            1834         0.317   1334
            regair           15493         0.262  12054
s_b1364_25  heliox            4949         0.464   2799
            regair            4911         0.480   2701
s_b1481h_24 heliox             461         0.115    462
            regair            1226         0.131   1142
s_b1484h_24 heliox            2026         0.280   1539
            regair            3959         0.226   3185
s_b1556h_24 heliox            1053         0.134    937
            regair            4054         0.126   3633
transitions: complete (48653 rows)
runs: complete (37343 rows)

saved /mnt/cube/lstanwic/umap_avgn_